In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Core libraries used for API requests and environment variables
import os
import requests

from datetime import datetime, timezone

# Load variables stored inside the .env file
from dotenv import load_dotenv

# Load the .env file from the project root
load_dotenv()

print("Libraries imported successfully.")

Libraries imported successfully.


### Weather Data Collection using OpenWeather

In [3]:
# Retrieve the OpenWeather API key from the .env file.
# The actual key is never written directly inside the notebook.
WEATHER_API_KEY = os.getenv("WEATHER_API_KEY")

# Check whether the key was successfully loaded.
# We only print whether it exists, NOT the actual secret key.
if WEATHER_API_KEY:
    print("OpenWeather API key loaded successfully.")
else:
    print("OpenWeather API key was not found.")

OpenWeather API key loaded successfully.


In [4]:
# Test destination used only to verify that the API connection works.
city = "Hyderabad"

# OpenWeather current-weather endpoint.
url = "https://api.openweathermap.org/data/2.5/weather"

# Parameters sent to the API.
params = {
    "q": city,
    "appid": WEATHER_API_KEY,
    "units": "metric"
}

# Send the request to OpenWeather.
response = requests.get(url, params=params, timeout=10)

# Display the HTTP status code.
print("Status code:", response.status_code)

# Display a small part of the response to confirm that data was received.
print(response.json())

Status code: 200
{'coord': {'lon': 78.4744, 'lat': 17.3753}, 'weather': [{'id': 801, 'main': 'Clouds', 'description': 'few clouds', 'icon': '02n'}], 'base': 'stations', 'main': {'temp': 26.23, 'feels_like': 26.23, 'temp_min': 25.73, 'temp_max': 26.23, 'pressure': 1010, 'humidity': 73, 'sea_level': 1010, 'grnd_level': 945}, 'visibility': 10000, 'wind': {'speed': 5.14, 'deg': 270}, 'clouds': {'all': 22}, 'dt': 1787668189, 'sys': {'type': 1, 'id': 9214, 'country': 'IN', 'sunrise': 1787617872, 'sunset': 1787663151}, 'timezone': 19800, 'id': 1269843, 'name': 'Hyderabad', 'cod': 200}


In [5]:
# Convert the API response into a Python dictionary.
weather_data = response.json()

# Display only the top-level fields returned by OpenWeather.
print(weather_data.keys())

dict_keys(['coord', 'weather', 'base', 'main', 'visibility', 'wind', 'clouds', 'dt', 'sys', 'timezone', 'id', 'name', 'cod'])


In [56]:
def get_weather(city):
    """
    Fetch current weather information for a given city.

    Parameters
    ----------
    city : str
        Name of the destination city.

    Returns
    -------
    dict or None
        A cleaned dictionary containing the weather features
        required for our Travel Agent project.
        Returns None if the API request fails.
    """

    # OpenWeather current weather endpoint
    url = "https://api.openweathermap.org/data/2.5/weather"

    # Parameters required by the API
    params = {
        "q": city,
        "appid": WEATHER_API_KEY,
        "units": "metric"
    }

    try:
        # Send request to OpenWeather
        response = requests.get(url, params=params, timeout=10)

        # Raise an exception automatically for HTTP errors
        response.raise_for_status()

        # Convert JSON response into a Python dictionary
        data = response.json()

        # Extract and return only the fields needed by our project
        weather = {
            "destination": data["name"],
            "country": data["sys"]["country"],
            "latitude": data["coord"]["lat"],
            "longitude": data["coord"]["lon"],
            "temperature": data["main"]["temp"],
            "feels_like": data["main"]["feels_like"],
            "humidity": data["main"]["humidity"],
            "pressure": data["main"]["pressure"],
            "wind_speed": data["wind"]["speed"],
            "cloudiness": data["clouds"]["all"],
            "weather_condition": data["weather"][0]["main"],
            "weather_description": data["weather"][0]["description"],
            "visibility": data.get("visibility"),
            "rain_1h": data.get("rain", {}).get("1h", 0),
            # Convert Unix timestamp returned by OpenWeather
            # into a readable UTC datetime.
            "timestamp": datetime.fromtimestamp(
                data["dt"],
                tz=timezone.utc
            ).isoformat()
        }

        return weather

    except requests.exceptions.RequestException as e:
        # Handle connection, timeout, HTTP, and API errors
        print(f"API request failed for {city}: {e}")
        return None

    except KeyError as e:
        # Handle unexpected/missing fields in the API response
        print(f"Unexpected API response for {city}. Missing field: {e}")
        return None

In [7]:
# Test the reusable weather function with Hyderabad
hyderabad_weather = get_weather("Hyderabad")

# Display the cleaned result
hyderabad_weather

{'destination': 'Hyderabad',
 'country': 'IN',
 'latitude': 17.3753,
 'longitude': 78.4744,
 'temperature': 26.23,
 'feels_like': 26.23,
 'humidity': 73,
 'pressure': 1010,
 'wind_speed': 5.14,
 'cloudiness': 22,
 'weather_condition': 'Clouds',
 'weather_description': 'few clouds',
 'visibility': 10000,
 'rain_1h': 0,
 'timestamp': '2026-08-25T14:29:49+00:00'}

In [8]:
# Initial destinations used to test our API pipeline.
# We will finalize the complete destination list later.
test_destinations = [
    "Hyderabad",
    "Goa",
    "Munnar",
    "Manali",
    "Jaipur"
]

# Store the API responses for each destination.
weather_results = []

for city in test_destinations:
    print(f"Fetching weather for {city}...")

    weather = get_weather(city)

    if weather is not None:
        weather_results.append(weather)
        print("  ✓ Success")
    else:
        print("  ✗ Failed")

print(f"\nSuccessfully collected: {len(weather_results)} / {len(test_destinations)}")

Fetching weather for Hyderabad...
  ✓ Success
Fetching weather for Goa...
  ✓ Success
Fetching weather for Munnar...
  ✓ Success
Fetching weather for Manali...
  ✓ Success
Fetching weather for Jaipur...
  ✓ Success

Successfully collected: 5 / 5


In [9]:
import pandas as pd

# Convert the list of dictionaries returned by the API
# into a structured pandas DataFrame.
weather_df = pd.DataFrame(weather_results)

# Display the collected data.
weather_df

,destination,country,latitude,longitude,temperature,feels_like,humidity,pressure,wind_speed,cloudiness,weather_condition,weather_description,visibility,rain_1h,timestamp
0,Hyderabad,IN,17.3753,78.4744,26.23,26.23,73,1010,5.14,22,Clouds,few clouds,10000.0,0.00,2026-08-25T14:29:49+00:00
1,Goa,IN,15.3333,74.0833,26.21,26.21,92,1010,1.96,66,Rain,light rain,10000.0,0.23,2026-08-25T14:28:43+00:00
2,Munnar,IN,10.1000,77.0667,16.23,16.42,96,1016,2.11,100,Clouds,overcast clouds,NaN,0.00,2026-08-25T14:28:58+00:00
3,Manali,IN,13.1667,80.2667,30.29,37.29,79,1007,0.45,98,Clouds,overcast clouds,10000.0,0.00,2026-08-25T14:30:02+00:00
4,Jaipur,IN,26.9167,75.8167,27.62,32.42,89,1004,0.00,97,Clouds,overcast clouds,10000.0,0.00,2026-08-25T14:26:24+00:00


In [10]:
from pathlib import Path

# Define the directory where raw weather API responses will be stored.
raw_weather_dir = Path("../data/raw/weather")

# Create the directory if it does not already exist.
raw_weather_dir.mkdir(parents=True, exist_ok=True)

# Save the current API test data as a CSV snapshot.
test_weather_path = raw_weather_dir / "weather_test.csv"

weather_df.to_csv(test_weather_path, index=False)

print(f"Test weather data saved to: {test_weather_path}")

Test weather data saved to: ..\data\raw\weather\weather_test.csv


### Destination Data Collection

In [11]:
# Initial master list of Indian tourist destinations.
# This list acts as the common reference for all APIs in our project.

destinations = [
    "Goa",
    "Munnar",
    "Manali",
    "Jaipur",
    "Udaipur",
    "Jaisalmer",
    "Jodhpur",
    "Agra",
    "Varanasi",
    "Rishikesh",
    "Shimla",
    "Mussoorie",
    "Nainital",
    "Darjeeling",
    "Gangtok",
    "Ooty",
    "Kodaikanal",
    "Coorg",
    "Wayanad",
    "Alappuzha",
    "Kochi",
    "Thiruvananthapuram",
    "Varkala",
    "Pondicherry",
    "Mahabalipuram",
    "Hampi",
    "Mysore",
    "Gokarna",
    "Andaman",
    "Mumbai",
    "Delhi",
    "Amritsar",
    "Ladakh",
    "Srinagar",
    "Dharamshala",
    "Kolkata",
    "Bengaluru",
    "Hyderabad",
    "Chennai",
    "Pune",
    "Ahmedabad",
    "Bhopal",
    "Indore",
    "Ranchi",
    "Bhubaneswar",
    "Shillong",
    "Kaziranga",
    "Jim Corbett",
    "Ranthambore",
    "Pahalgam"
]

# Convert the list into a DataFrame.
destination_master = pd.DataFrame({
    "destination": destinations
})

print(f"Number of destinations: {len(destination_master)}")

destination_master.head(10)

Number of destinations: 50


,destination
0,Goa
1,Munnar
2,Manali
3,Jaipur
4,Udaipur
5,Jaisalmer
6,Jodhpur
7,Agra
8,Varanasi
9,Rishikesh


In [12]:
# Create a stable numeric ID for each destination.
# This ID will make it easier to join data from different APIs later.

destination_master.insert(
    0,
    "destination_id",
    range(1, len(destination_master) + 1)
)

destination_master.head()

,destination_id,destination
0,1,Goa
1,2,Munnar
2,3,Manali
3,4,Jaipur
4,5,Udaipur


In [13]:
# Save the destination master list in the raw data directory.
# This file will become the common reference for our API pipeline.

destination_master_path = Path("../data/raw/destination_master.csv")

destination_master.to_csv(
    destination_master_path,
    index=False
)

print(f"Destination master saved to: {destination_master_path}")

Destination master saved to: ..\data\raw\destination_master.csv


In [14]:
# Load the Geoapify API key from the .env file.
# We never place the actual secret key directly in the notebook.

GEOAPIFY_API_KEY = os.getenv("GEOAPIFY_API_KEY")

if GEOAPIFY_API_KEY:
    print("Geoapify API key loaded successfully.")
else:
    print("Geoapify API key was not found.")

Geoapify API key loaded successfully.


In [15]:
# Test destination for the Geoapify geocoding API.
city = "Goa"

# Geoapify geocoding endpoint.
url = "https://api.geoapify.com/v1/geocode/search"

# Parameters sent to Geoapify.
params = {
    "text": city,
    "apiKey": GEOAPIFY_API_KEY,
    "limit": 1
}

# Send the request.
response = requests.get(
    url,
    params=params,
    timeout=10
)

print("Status code:", response.status_code)

# Display the API response.
print(response.json())

Status code: 200
{'type': 'FeatureCollection', 'features': [{'type': 'Feature', 'properties': {'country': 'India', 'country_code': 'in', 'state': 'Goa', 'name': 'Goa', 'other_names': {'name': 'Goa', 'name:cs': 'Goa', 'name:en': 'Goa', 'name:eo': 'Goao', 'name:he': 'גואה', 'name:hi': 'गोवा', 'name:ja': 'ゴア', 'name:kn': 'ಗೋವಾ', 'name:ko': '고아', 'name:ku': 'Goa', 'name:mr': 'गोआ', 'name:pa': 'ਗੋਆ', 'name:pt': 'Goa', 'name:ru': 'Гоа', 'name:ta': 'கோவா', 'name:uk': 'Ґоа', 'name:zh': '果阿邦', 'name:gom': 'गोंय', 'ISO3166-2': 'IN-GA', '_place_ref': 'GA', '_place_name:fa': 'گوا', '_place_name:hu': 'Goa', '_place_name:kn': 'ಗೋವ', '_place_name:ml': 'ഗോവ', '_place_name:te': 'గోవా', '_place_name:kok': 'गोंय'}, 'datasource': {'sourcename': 'openstreetmap', 'attribution': '© OpenStreetMap contributors', 'license': 'Open Database License', 'url': 'https://www.openstreetmap.org/copyright'}, 'state_code': 'GA', 'iso3166_2': 'IN-GA', 'lon': 74.0855134, 'lat': 15.3004543, 'result_type': 'state', 'formatted

In [16]:
# Convert the response to a Python dictionary.
geo_data = response.json()

# Inspect the top-level structure.
print(geo_data.keys())

dict_keys(['type', 'features', 'query'])


In [17]:
# Inspect the first result returned by Geoapify.
geo_data["features"][0]

{'type': 'Feature',
 'properties': {'country': 'India',
  'country_code': 'in',
  'state': 'Goa',
  'name': 'Goa',
  'other_names': {'name': 'Goa',
   'name:cs': 'Goa',
   'name:en': 'Goa',
   'name:eo': 'Goao',
   'name:he': 'גואה',
   'name:hi': 'गोवा',
   'name:ja': 'ゴア',
   'name:kn': 'ಗೋವಾ',
   'name:ko': '고아',
   'name:ku': 'Goa',
   'name:mr': 'गोआ',
   'name:pa': 'ਗੋਆ',
   'name:pt': 'Goa',
   'name:ru': 'Гоа',
   'name:ta': 'கோவா',
   'name:uk': 'Ґоа',
   'name:zh': '果阿邦',
   'name:gom': 'गोंय',
   'ISO3166-2': 'IN-GA',
   '_place_ref': 'GA',
   '_place_name:fa': 'گوا',
   '_place_name:hu': 'Goa',
   '_place_name:kn': 'ಗೋವ',
   '_place_name:ml': 'ഗോവ',
   '_place_name:te': 'గోవా',
   '_place_name:kok': 'गोंय'},
  'datasource': {'sourcename': 'openstreetmap',
   'attribution': '© OpenStreetMap contributors',
   'license': 'Open Database License',
   'url': 'https://www.openstreetmap.org/copyright'},
  'state_code': 'GA',
  'iso3166_2': 'IN-GA',
  'lon': 74.0855134,
  'lat': 15.

In [18]:
def get_location(destination):
    """
    Fetch geographic information for a destination using Geoapify.

    Parameters
    ----------
    destination : str
        Destination name to search.

    Returns
    -------
    dict or None
        Clean geographic information including coordinates,
        state, country, result type, and Geoapify place ID.
        Returns None if the API request fails or no result is found.
    """

    # Geoapify geocoding endpoint
    url = "https://api.geoapify.com/v1/geocode/search"

    # Parameters sent to the API
    params = {
        "text": destination,
        "apiKey": GEOAPIFY_API_KEY,
        "limit": 1
    }

    try:
        # Send the request to Geoapify
        response = requests.get(
            url,
            params=params,
            timeout=10
        )

        # Raise an exception for HTTP errors
        response.raise_for_status()

        # Convert the JSON response into a Python dictionary
        data = response.json()

        # Check whether Geoapify returned any locations
        if not data.get("features"):
            print(f"No location found for {destination}")
            return None

        # Extract the highest-ranked result
        properties = data["features"][0]["properties"]

        # Return only the fields required by our project
        location = {
            "destination": destination,
            "resolved_name": properties.get("name"),
            "country": properties.get("country"),
            "country_code": properties.get("country_code"),
            "state": properties.get("state"),
            "state_code": properties.get("state_code"),
            "latitude": properties.get("lat"),
            "longitude": properties.get("lon"),
            "result_type": properties.get("result_type"),
            "formatted_address": properties.get("formatted"),
            "confidence": properties.get("rank", {}).get("confidence"),
            "match_type": properties.get("rank", {}).get("match_type"),
            "place_id": properties.get("place_id")
        }

        return location

    except requests.exceptions.RequestException as e:
        # Handle connection, timeout, and HTTP errors
        print(f"API request failed for {destination}: {e}")
        return None

    except (KeyError, TypeError) as e:
        # Handle unexpected API response structures
        print(f"Unexpected response for {destination}: {e}")
        return None

In [19]:
# Test the reusable Geoapify location function.
goa_location = get_location("Goa")

# Display the cleaned result.
goa_location

{'destination': 'Goa',
 'resolved_name': 'Goa',
 'country': 'India',
 'country_code': 'in',
 'state': 'Goa',
 'state_code': 'GA',
 'latitude': 15.3004543,
 'longitude': 74.0855134,
 'result_type': 'state',
 'formatted_address': 'Goa, India',
 'confidence': 1,
 'match_type': 'full_match',
 'place_id': '51aa17320d798552405999e26025d5992e40f00101f90125afab0000000000c0020a920303476f61'}

In [20]:
test_locations = [
    "Goa",          # State
    "Munnar",       # Town
    "Manali",       # Town
    "Delhi",        # City
    "Ladakh",       # Region
    "Jim Corbett",  # National park
    "Ranthambore",  # National park/region
    "Andaman"       # Island/region
]

for destination in test_locations:

    location = get_location(destination)

    if location:
        print(
            f"{destination:15} → "
            f"{location['resolved_name']} | "
            f"{location['result_type']} | "
            f"{location['state']}"
        )
    else:
        print(f"{destination:15} → FAILED")

Goa             → Goa | state | Goa
Munnar          → Munnar | city | Kerala
Manali          → Manali | city | Himachal Pradesh
Delhi           → Delhi wala | amenity | Telangana
Ladakh          → Ladakh | state | Ladakh
Jim Corbett     → Jim Corbett National Park | amenity | Uttarakhand
Ranthambore     → Ranthambore Fort | amenity | Rajasthan
Andaman         → Andaman and Nicobar Islands | state | Andaman and Nicobar Islands


In [21]:
# Test corrected search queries for destinations
# that were ambiguous in the first Geoapify request.

corrected_queries = {
    "Delhi": "Delhi, India",
    "Ranthambore": "Ranthambore National Park, Rajasthan, India"
}

for destination, search_query in corrected_queries.items():

    location = get_location(search_query)

    if location:
        print(
            f"{destination:15} → "
            f"{location['resolved_name']} | "
            f"{location['result_type']} | "
            f"{location['state']} | "
            f"{location['latitude']}, {location['longitude']}"
        )
    else:
        print(f"{destination:15} → FAILED")

Delhi           → New Delhi | city | None | 28.6138954, 77.2090057
Ranthambore     → Bandhavgarh National Park | street | Madhya Pradesh | 23.717931, 81.024416


In [22]:
# Test Sawai Madhopur as the geographic anchor for Ranthambore.
# This avoids Geoapify confusing "Ranthambore" with another park.

ranthambore_query = "Sawai Madhopur, Rajasthan, India"

ranthambore_location = get_location(ranthambore_query)

ranthambore_location

{'destination': 'Sawai Madhopur, Rajasthan, India',
 'resolved_name': 'Sawai Madhopur',
 'country': 'India',
 'country_code': 'in',
 'state': 'Rajasthan',
 'state_code': 'RJ',
 'latitude': 26.0188919,
 'longitude': 76.3522441,
 'result_type': 'city',
 'formatted_address': 'Sawai Madhopur, RJ, India',
 'confidence': 1,
 'match_type': 'full_match',
 'place_id': '515f6dd62a8b16534059c8a87c19d6043a40f00103f901bc5b3a5600000000c0020892030e5361776169204d6164686f707572'}

In [23]:
# Create a copy so that our original destination list remains unchanged.
destination_master = destination_master.copy()

# By default, use the destination name itself as the API search query.
destination_master["search_query"] = destination_master["destination"]

# Override destinations where a direct search can produce an incorrect
# or ambiguous geographic result.
destination_master.loc[
    destination_master["destination"] == "Delhi",
    "search_query"
] = "Delhi, India"

destination_master.loc[
    destination_master["destination"] == "Ranthambore",
    "search_query"
] = "Sawai Madhopur, Rajasthan, India"

# Display the mappings we explicitly corrected.
destination_master[
    destination_master["destination"].isin(["Delhi", "Ranthambore"])
]

,destination_id,destination,search_query
30,31,Delhi,"Delhi, India"
48,49,Ranthambore,"Sawai Madhopur, Rajasthan, India"


In [24]:
# Save the updated destination master.
# The search_query column tells each API how to locate the destination.

destination_master.to_csv(
    "../data/raw/destination_master.csv",
    index=False
)

print("Updated destination master saved successfully.")
print(f"Total destinations: {len(destination_master)}")

Updated destination master saved successfully.
Total destinations: 50


In [25]:
destination_master.head(10)

,destination_id,destination,search_query
0,1,Goa,Goa
1,2,Munnar,Munnar
2,3,Manali,Manali
3,4,Jaipur,Jaipur
4,5,Udaipur,Udaipur
5,6,Jaisalmer,Jaisalmer
6,7,Jodhpur,Jodhpur
7,8,Agra,Agra
8,9,Varanasi,Varanasi
9,10,Rishikesh,Rishikesh


In [26]:
# Collect geographic information for all 50 destinations.
# We use search_query rather than destination so that our corrected
# API mappings are respected.

location_results = []

for _, row in destination_master.iterrows():

    destination = row["destination"]
    search_query = row["search_query"]

    print(f"Searching: {destination} → {search_query}")

    location = get_location(search_query)

    if location is not None:

        # Keep the original destination identity separate from the
        # API's resolved geographic name.
        location["destination_id"] = row["destination_id"]
        location["destination"] = destination
        location["search_query"] = search_query

        location_results.append(location)

        print(
            f"  ✓ {location['resolved_name']} | "
            f"{location['result_type']} | "
            f"{location['state']}"
        )

    else:
        print("  ✗ No valid result")

print(
    f"\nSuccessfully resolved: "
    f"{len(location_results)} / {len(destination_master)}"
)

Searching: Goa → Goa


  ✓ Goa | state | Goa
Searching: Munnar → Munnar
  ✓ Munnar | city | Kerala
Searching: Manali → Manali
  ✓ Manali | city | Himachal Pradesh
Searching: Jaipur → Jaipur
  ✓ Jaipur | city | Rajasthan
Searching: Udaipur → Udaipur
  ✓ Udaipur | city | Rajasthan
Searching: Jaisalmer → Jaisalmer
  ✓ Jaisalmer | city | Rajasthan
Searching: Jodhpur → Jodhpur
  ✓ Jodhpur | city | Rajasthan
Searching: Agra → Agra
  ✓ Agra | city | Uttar Pradesh
Searching: Varanasi → Varanasi
  ✓ Varanasi | city | Uttar Pradesh
Searching: Rishikesh → Rishikesh
  ✓ Rishikesh | city | Uttarakhand
Searching: Shimla → Shimla
  ✓ Shimla | city | Himachal Pradesh
Searching: Mussoorie → Mussoorie
  ✓ Mussoorie | city | Uttarakhand
Searching: Nainital → Nainital
  ✓ Nainital | county | Uttarakhand
Searching: Darjeeling → Darjeeling
  ✓ Darjeeling | city | West Bengal
Searching: Gangtok → Gangtok
  ✓ Gangtok | city | Sikkim
Searching: Ooty → Ooty
  ✓ Udhagamandalam | city | Tamil Nadu
Searching: Kodaikanal → Kodaikanal
  ✓

In [27]:
# Convert the API results into a DataFrame for inspection.
location_df = pd.DataFrame(location_results)

location_df[
    [
        "destination_id",
        "destination",
        "search_query",
        "resolved_name",
        "state",
        "result_type",
        "latitude",
        "longitude",
        "confidence",
        "match_type"
    ]
]

,destination_id,destination,search_query,resolved_name,state,result_type,latitude,longitude,confidence,match_type
0,1,Goa,Goa,Goa,Goa,state,15.300454,74.085513,1.000000,full_match
1,2,Munnar,Munnar,Munnar,Kerala,city,10.086996,77.060091,1.000000,full_match
2,3,Manali,Manali,Manali,Himachal Pradesh,city,32.245461,77.187293,1.000000,full_match
3,4,Jaipur,Jaipur,Jaipur,Rajasthan,city,26.915458,75.818982,1.000000,full_match
4,5,Udaipur,Udaipur,Udaipur,Rajasthan,city,24.578721,73.686257,1.000000,full_match
5,6,Jaisalmer,Jaisalmer,Jaisalmer,Rajasthan,city,26.911662,70.912489,1.000000,full_match
6,7,Jodhpur,Jodhpur,Jodhpur,Rajasthan,city,26.296772,73.035143,1.000000,full_match
7,8,Agra,Agra,Agra,Uttar Pradesh,city,27.175255,78.009816,1.000000,full_match
8,9,Varanasi,Varanasi,Varanasi,Uttar Pradesh,city,25.335649,83.007629,1.000000,full_match
9,10,Rishikesh,Rishikesh,Rishikesh,Uttarakhand,city,30.108654,78.291619,1.000000,full_match


In [28]:
# Correct ambiguous destination search queries identified
# during the 50-destination Geoapify validation.

destination_master.loc[
    destination_master["destination"] == "Nainital",
    "search_query"
] = "Nainital, Uttarakhand, India"

destination_master.loc[
    destination_master["destination"] == "Ooty",
    "search_query"
] = "Ooty, Tamil Nadu, India"

destination_master.loc[
    destination_master["destination"] == "Kochi",
    "search_query"
] = "Kochi, Kerala, India"

destination_master.loc[
    destination_master["destination"] == "Pondicherry",
    "search_query"
] = "Puducherry, India"

# Display the corrected mappings.
destination_master[
    destination_master["destination"].isin([
        "Nainital",
        "Ooty",
        "Kochi",
        "Pondicherry"
    ])
][["destination", "search_query"]]

,destination,search_query
12,Nainital,"Nainital, Uttarakhand, India"
15,Ooty,"Ooty, Tamil Nadu, India"
20,Kochi,"Kochi, Kerala, India"
23,Pondicherry,"Puducherry, India"


In [29]:
# Re-test only the destinations that had ambiguous or incorrect
# Geoapify results in the first validation pass.

correction_tests = [
    "Nainital",
    "Ooty",
    "Kochi",
    "Pondicherry"
]

for destination in correction_tests:

    search_query = destination_master.loc[
        destination_master["destination"] == destination,
        "search_query"
    ].iloc[0]

    location = get_location(search_query)

    if location:
        print(
            f"{destination:15} → "
            f"{location['resolved_name']} | "
            f"{location['state']} | "
            f"{location['result_type']} | "
            f"{location['latitude']}, {location['longitude']} | "
            f"confidence={location['confidence']}"
        )
    else:
        print(f"{destination:15} → FAILED")

Nainital        → Nainital | Uttarakhand | city | 29.3905295, 79.460869 | confidence=1
Ooty            → Udhagamandalam | Tamil Nadu | city | 11.4126769, 76.7030504 | confidence=0
Kochi           → Kochi | Kerala | city | 9.9679032, 76.2444378 | confidence=1
Pondicherry     → Puducherry | Puducherry | city | 11.9340568, 79.8306447 | confidence=1


In [30]:
# Re-run geocoding for all 50 destinations using the corrected
# search_query values from destination_master.
#
# This creates one consistent, validated location snapshot
# after all ambiguous destinations have been fixed.

validated_locations = []

for _, row in destination_master.iterrows():

    destination_id = row["destination_id"]
    destination = row["destination"]
    search_query = row["search_query"]

    print(f"Collecting: {destination}")

    location = get_location(search_query)

    if location is not None:

        # Keep the project's original destination identity.
        location["destination_id"] = destination_id
        location["destination"] = destination
        location["search_query"] = search_query

        validated_locations.append(location)

    else:
        print(f"  ✗ Failed: {destination}")

print(
    f"\nSuccessfully collected: "
    f"{len(validated_locations)} / {len(destination_master)}"
)

Collecting: Goa
Collecting: Munnar
Collecting: Manali
Collecting: Jaipur
Collecting: Udaipur
Collecting: Jaisalmer
Collecting: Jodhpur
Collecting: Agra
Collecting: Varanasi
Collecting: Rishikesh
Collecting: Shimla
Collecting: Mussoorie
Collecting: Nainital
Collecting: Darjeeling
Collecting: Gangtok
Collecting: Ooty
Collecting: Kodaikanal
Collecting: Coorg
Collecting: Wayanad
Collecting: Alappuzha
Collecting: Kochi
Collecting: Thiruvananthapuram
Collecting: Varkala
Collecting: Pondicherry
Collecting: Mahabalipuram
Collecting: Hampi
Collecting: Mysore
Collecting: Gokarna
Collecting: Andaman
Collecting: Mumbai
Collecting: Delhi
Collecting: Amritsar
Collecting: Ladakh
Collecting: Srinagar
Collecting: Dharamshala
Collecting: Kolkata
Collecting: Bengaluru
Collecting: Hyderabad
Collecting: Chennai
Collecting: Pune
Collecting: Ahmedabad
Collecting: Bhopal
Collecting: Indore
Collecting: Ranchi
Collecting: Bhubaneswar


KeyboardInterrupt: 

In [ ]:
# Convert the validated API results into a DataFrame.
validated_location_df = pd.DataFrame(validated_locations)

# Arrange the columns in a logical order for our project.
validated_location_df = validated_location_df[
    [
        "destination_id",
        "destination",
        "search_query",
        "resolved_name",
        "country",
        "country_code",
        "state",
        "state_code",
        "latitude",
        "longitude",
        "result_type",
        "formatted_address",
        "confidence",
        "match_type",
        "place_id"
    ]
]

validated_location_df.head()

,destination_id,destination,search_query,resolved_name,country,country_code,state,state_code,latitude,longitude,result_type,formatted_address,confidence,match_type,place_id
0,1,Goa,Goa,Goa,India,in,Goa,GA,15.300454,74.085513,state,"Goa, India",1.0,full_match,51aa17320d798552405999e26025d5992e40f00101f901...
1,2,Munnar,Munnar,Munnar,India,in,Kerala,KL,10.086996,77.060091,city,"Munnar, KL, India",1.0,full_match,5120d1048ad843534059adc502ba8a2c2440f00103f901...
2,3,Manali,Manali,Manali,India,in,Himachal Pradesh,HP,32.245461,77.187293,city,"Manali, HP, India",1.0,full_match,511af2199afc4b53405999396e426b1f4040f00103f901...
3,4,Jaipur,Jaipur,Jaipur,India,in,Rajasthan,RJ,26.915458,75.818982,city,"Jaipur, RJ, India",1.0,full_match,51706138326af4524059e9dfe46d5bea3a40f00103f901...
4,5,Udaipur,Udaipur,Udaipur,India,in,Rajasthan,RJ,24.578721,73.686257,city,"Udaipur, RJ, India",1.0,full_match,517649e6a2eb6b5240592882380f27943840f00103f901...


In [ ]:
# Basic validation checks before saving the location dataset.

print("Number of rows:", len(validated_location_df))

print(
    "Unique destinations:",
    validated_location_df["destination"].nunique()
)

print(
    "Missing latitude:",
    validated_location_df["latitude"].isna().sum()
)

print(
    "Missing longitude:",
    validated_location_df["longitude"].isna().sum()
)

print(
    "Missing country:",
    validated_location_df["country"].isna().sum()
)

Number of rows: 50
Unique destinations: 50
Missing latitude: 0
Missing longitude: 0
Missing country: 0


In [ ]:
# Save the validated geographic dataset.
# This becomes the geographic foundation for the rest of our API pipeline.

location_output_path = Path(
    "../data/raw/places/destination_locations.csv"
)

# Ensure the places directory exists.
location_output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

validated_location_df.to_csv(
    location_output_path,
    index=False
)

print(
    f"Validated location dataset saved to: "
    f"{location_output_path}"
)

Validated location dataset saved to: ..\data\raw\places\destination_locations.csv


In [ ]:
# Get Goa's validated coordinates from our location dataset.
# Using coordinates avoids ambiguous text-based searches.

goa = validated_location_df[
    validated_location_df["destination"] == "Goa"
].iloc[0]

goa_latitude = goa["latitude"]
goa_longitude = goa["longitude"]

print("Goa latitude:", goa_latitude)
print("Goa longitude:", goa_longitude)

Goa latitude: 15.3004543
Goa longitude: 74.0855134


In [ ]:
# Geoapify Places API endpoint.
places_url = "https://api.geoapify.com/v2/places"

# Search for tourism-related places around Goa.
places_params = {
    "categories": "tourism",
    "filter": f"circle:{goa_longitude},{goa_latitude},20000",
    "limit": 20,
    "apiKey": GEOAPIFY_API_KEY
}

# Send the request.
places_response = requests.get(
    places_url,
    params=places_params,
    timeout=10
)

print("Status code:", places_response.status_code)

# Inspect the returned JSON.
places_data = places_response.json()

print(places_data)

Status code: 200
{'type': 'FeatureCollection', 'features': [{'type': 'Feature', 'properties': {'country': 'India', 'country_code': 'in', 'state': 'Goa', 'county': 'Quepem', 'state_district': 'Kushavati', 'city': 'Acamor', 'postcode': '403703', 'street': 'Acamol Road', 'iso3166_2': 'IN-GA', 'lon': 74.0371788, 'lat': 15.1938711, 'state_code': 'GA', 'formatted': 'Acamol Road, Acamor - 403703, Goa, India', 'address_line1': 'Acamol Road', 'address_line2': 'Acamor - 403703, Goa, India', 'categories': ['tourism', 'tourism.attraction', 'tourism.attraction.viewpoint'], 'details': [], 'datasource': {'sourcename': 'openstreetmap', 'attribution': '© OpenStreetMap contributors', 'license': 'Open Database License', 'url': 'https://www.openstreetmap.org/copyright', 'raw': {'lat': 15.1938711, 'lon': 74.0371788, 'osm_id': 4826443350, 'tourism': 'viewpoint', 'osm_type': 'n'}}, 'place_id': '51b186302361825240597347a41243632e40f00103f90156aead1f01000000'}, 'geometry': {'type': 'Point', 'coordinates': [74.

In [ ]:
# Check how many places were returned.
print("Number of places returned:", len(places_data.get("features", [])))

Number of places returned: 20


In [ ]:
# Inspect the first returned place to understand the actual
# Geoapify response structure before writing our extraction function.

if places_data.get("features"):
    print(places_data["features"][0])
else:
    print("No places returned.")

{'type': 'Feature', 'properties': {'country': 'India', 'country_code': 'in', 'state': 'Goa', 'county': 'Quepem', 'state_district': 'Kushavati', 'city': 'Acamor', 'postcode': '403703', 'street': 'Acamol Road', 'iso3166_2': 'IN-GA', 'lon': 74.0371788, 'lat': 15.1938711, 'state_code': 'GA', 'formatted': 'Acamol Road, Acamor - 403703, Goa, India', 'address_line1': 'Acamol Road', 'address_line2': 'Acamor - 403703, Goa, India', 'categories': ['tourism', 'tourism.attraction', 'tourism.attraction.viewpoint'], 'details': [], 'datasource': {'sourcename': 'openstreetmap', 'attribution': '© OpenStreetMap contributors', 'license': 'Open Database License', 'url': 'https://www.openstreetmap.org/copyright', 'raw': {'lat': 15.1938711, 'lon': 74.0371788, 'osm_id': 4826443350, 'tourism': 'viewpoint', 'osm_type': 'n'}}, 'place_id': '51b186302361825240597347a41243632e40f00103f90156aead1f01000000'}, 'geometry': {'type': 'Point', 'coordinates': [74.0371788, 15.1938711]}}


## The Function is for getting the places

In [ ]:
def get_places(latitude, longitude, radius=20000, limit=100):
    """
    Fetch tourist places around a geographic coordinate using Geoapify.

    Parameters
    ----------
    latitude : float
        Latitude of the destination.

    longitude : float
        Longitude of the destination.

    radius : int
        Search radius in meters.
        Default is 20 km.

    limit : int
        Maximum number of places requested from the API.

    Returns
    -------
    list
        A list of cleaned place dictionaries.
        Returns an empty list if the request fails.
    """

    # Geoapify Places API endpoint.
    url = "https://api.geoapify.com/v2/places"

    # API request parameters.
    params = {
        "categories": "tourism",
        "filter": f"circle:{longitude},{latitude},{radius}",
        "limit": limit,
        "apiKey": GEOAPIFY_API_KEY
    }

    try:
        # Send the request to Geoapify.
        response = requests.get(
            url,
            params=params,
            timeout=10
        )

        # Raise an exception if the API returns an HTTP error.
        response.raise_for_status()

        # Convert the JSON response into a Python dictionary.
        data = response.json()

        # Store cleaned places here.
        places = []

        # Process every returned feature.
        for feature in data.get("features", []):

            properties = feature.get("properties", {})

            # Extract the geographic coordinates.
            geometry = feature.get("geometry", {})
            coordinates = geometry.get("coordinates", [None, None])

            # GeoJSON coordinates are [longitude, latitude].
            place_longitude = coordinates[0]
            place_latitude = coordinates[1]

            # Extract categories returned by Geoapify.
            categories = properties.get("categories", [])

            # Store only fields useful for our project.
            place = {
                "name": properties.get("name"),
                "country": properties.get("country"),
                "state": properties.get("state"),
                "city": properties.get("city"),
                "latitude": place_latitude,
                "longitude": place_longitude,
                "categories": categories,
                "formatted_address": properties.get("formatted"),
                "place_id": properties.get("place_id")
            }

            places.append(place)

        return places

    except requests.exceptions.RequestException as e:
        # Handle connection, timeout, and HTTP errors.
        print(f"Places API request failed: {e}")
        return []

    except (KeyError, TypeError, IndexError) as e:
        # Handle unexpected response structures.
        print(f"Unexpected Places API response: {e}")
        return []

In [ ]:
# Test the reusable Places API function using Goa.

goa_places = get_places(
    latitude=goa_latitude,
    longitude=goa_longitude
)

print("Number of places collected:", len(goa_places))

# Display the first three cleaned records.
goa_places[:3]

Number of places collected: 59


[{'name': None,
  'country': 'India',
  'state': 'Goa',
  'city': 'Acamor',
  'latitude': 15.1938711,
  'longitude': 74.0371788,
  'categories': ['tourism',
   'tourism.attraction',
   'tourism.attraction.viewpoint'],
  'formatted_address': 'Acamol Road, Acamor - 403703, Goa, India',
  'place_id': '51b186302361825240597347a41243632e40f00103f90156aead1f01000000'},
 {'name': 'Nagesh Gardens',
  'country': 'India',
  'state': 'Goa',
  'city': 'Chandor',
  'latitude': 15.242618,
  'longitude': 74.03319,
  'categories': ['tourism', 'tourism.attraction'],
  'formatted_address': 'Nagesh Gardens, SH8, Chandor - 403714, Goa, India',
  'place_id': '517923f3c81f8252405984d72e6d387c2e40f00103f901f04aa6fc0000000092030e4e61676573682047617264656e73'},
 {'name': 'Mystic Meadows - A conservatory for Bees , Birds & Butterflies',
  'country': 'India',
  'state': 'Goa',
  'city': 'Savoi-Verem',
  'latitude': 15.4297676,
  'longitude': 74.012586,
  'categories': ['tourism', 'tourism.attraction'],
  'format

In [ ]:
# Convert the cleaned place records into a DataFrame
# so we can inspect the structure more easily.

goa_places_df = pd.DataFrame(goa_places)

print("Rows:", len(goa_places_df))
print("Columns:", goa_places_df.columns.tolist())

goa_places_df.head()

Rows: 59
Columns: ['name', 'country', 'state', 'city', 'latitude', 'longitude', 'categories', 'formatted_address', 'place_id']


,name,country,state,city,latitude,longitude,categories,formatted_address,place_id
0,NaN,India,Goa,Acamor,15.193871,74.037179,"[tourism, tourism.attraction, tourism.attracti...","Acamol Road, Acamor - 403703, Goa, India",51b186302361825240597347a41243632e40f00103f901...
1,Nagesh Gardens,India,Goa,Chandor,15.242618,74.033190,"[tourism, tourism.attraction]","Nagesh Gardens, SH8, Chandor - 403714, Goa, India",517923f3c81f8252405984d72e6d387c2e40f00103f901...
2,"Mystic Meadows - A conservatory for Bees , Bir...",India,Goa,Savoi-Verem,15.429768,74.012586,"[tourism, tourism.attraction]","Mystic Meadows - A conservatory for Bees , Bir...",51cc988235ce80524059c5c2b57f0adc2e40f00103f901...
3,Roadside Cross,India,Goa,Mulem,15.217447,74.064913,"[tourism, tourism.attraction, tourism.attracti...","Roadside Cross, MDR40, Mulem - 403714, Goa, India",51d4241d8a2784524059ab36493d556f2e40f00103f901...
4,Dr. B. R. Ambedkar,India,Goa,Curchorem,15.258479,74.105321,"[tourism, tourism.attraction, tourism.attracti...","Dr. B. R. Ambedkar Roundabout, Curchorem - 403...",51d5642195bd865240597a64bd6257842e40f00103f901...


In [ ]:
# Categories we want to evaluate for our recommendation system.
# We test them individually before designing the final feature schema.

place_categories = [
    "tourism.attraction",
    "tourism.sights",
    "leisure.park",
    "natural",
    "entertainment",
    "catering.restaurant"
]

In [ ]:
from collections import Counter

# Count every category appearing in the returned Goa places.
# A place can have multiple categories, so we flatten the category lists first.

category_counter = Counter()

for place in goa_places:
    category_counter.update(place["categories"])

# Convert the category counts into a DataFrame for easier inspection.
category_df = pd.DataFrame(
    category_counter.items(),
    columns=["category", "count"]
).sort_values(
    "count",
    ascending=False
)

category_df

,category,count
0,tourism,59
1,tourism.attraction,59
3,tourism.attraction.artwork,27
4,tourism.attraction.artwork.statue,6
5,building,3
6,building.tourism,3
2,tourism.attraction.viewpoint,2
8,tourism.sights,2
7,building.historic,1
9,tourism.sights.city_gate,1


In [ ]:
# Count places that do not have a usable name.

unnamed_count = goa_places_df["name"].isna().sum()

print("Total places:", len(goa_places_df))
print("Unnamed places:", unnamed_count)
print(
    "Unnamed percentage:",
    round(unnamed_count / len(goa_places_df) * 100, 2),
    "%"
)

Total places: 59
Unnamed places: 4
Unnamed percentage: 6.78 %


In [ ]:
# Check whether Geoapify returned duplicate place IDs.

duplicate_place_ids = goa_places_df["place_id"].duplicated().sum()

print("Duplicate place IDs:", duplicate_place_ids)

Duplicate place IDs: 0


In [ ]:
# Candidate Geoapify categories for the recommendation system.
#
# We will test these on Goa first to see which categories
# return useful real-world places.

test_categories = [
    "tourism",
    "tourism.sights",
    "leisure.park",
    "natural",
    "catering.restaurant"
]

print("Categories to test:")
for category in test_categories:
    print("-", category)

Categories to test:
- tourism
- tourism.sights
- leisure.park
- natural
- catering.restaurant


In [ ]:
def test_place_category(latitude, longitude, category, radius=20000, limit=50):
    """
    Test one Geoapify Places API category around a destination.

    This function is used only during API exploration.
    It helps us determine which categories produce useful
    data before we build the final collection pipeline.
    """

    url = "https://api.geoapify.com/v2/places"

    params = {
        "categories": category,
        "filter": f"circle:{longitude},{latitude},{radius}",
        "limit": limit,
        "apiKey": GEOAPIFY_API_KEY
    }

    try:
        response = requests.get(
            url,
            params=params,
            timeout=10
        )

        response.raise_for_status()

        data = response.json()

        return data.get("features", [])

    except requests.exceptions.RequestException as e:
        print(f"Request failed for {category}: {e}")
        return []

In [ ]:
# Test each candidate category using Goa's validated coordinates.

category_test_results = []

for category in test_categories:

    features = test_place_category(
        latitude=goa_latitude,
        longitude=goa_longitude,
        category=category
    )

    category_test_results.append({
        "category": category,
        "places_returned": len(features)
    })

# Display the results.
category_test_df = pd.DataFrame(category_test_results)

category_test_df

,category,places_returned
0,tourism,50
1,tourism.sights,9
2,leisure.park,28
3,natural,50
4,catering.restaurant,50


In [ ]:
# Inspect representative places from the three categories
# that need further evaluation.

categories_to_inspect = [
    "tourism.sights",
    "leisure.park",
    "natural",
    "catering.restaurant"
]

for category in categories_to_inspect:

    features = test_place_category(
        latitude=goa_latitude,
        longitude=goa_longitude,
        category=category,
        limit=10
    )

    print("\n" + "=" * 70)
    print(f"CATEGORY: {category}")
    print(f"Returned: {len(features)}")

    # Display the first five places for manual inspection.
    for feature in features[:5]:

        properties = feature.get("properties", {})

        print(
            f"- {properties.get('name')} | "
            f"{properties.get('categories')} | "
            f"{properties.get('formatted')}"
        )


CATEGORY: tourism.sights
Returned: 9
- Shri. Santadurga Kunkalikaran Prasann | ['tourism', 'tourism.sights', 'tourism.sights.archaeological_site'] | Shri. Santadurga Kunkalikaran Prasann, NH66, Cuncolim - 403603, Goa, India
- Palacio Do Deao | ['building', 'building.historic', 'tourism', 'tourism.sights', 'tourism.sights.manor'] | Palacio Do Deao, MDR40, Quepem - 403705, Goa, India
- Memorial For Goan Victims of 1918 Spanish Flu | ['tourism', 'tourism.sights', 'tourism.sights.memorial'] | Memorial For Goan Victims of 1918 Spanish Flu, Margaon Ponda State Highway, Fatorda, Margao - 403600, Goa, India
- Rivona Caves | ['tourism', 'tourism.sights', 'tourism.sights.archaeological_site'] | Rivona Caves, MDR36, Rivona - 403705, Goa, India
- Cave at Ishwarbhat | ['tourism', 'tourism.sights', 'tourism.sights.archaeological_site'] | Cave at Ishwarbhat, MDR25, Bhamalwada, Usgao - 403406, Goa, India

CATEGORY: leisure.park
Returned: 10
- Heritage Park | ['leisure', 'leisure.park'] | Heritage Par

In [ ]:
def get_places_by_categories(
    latitude,
    longitude,
    categories,
    radius=20000,
    limit=100
):
    """
    Collect places from multiple Geoapify categories.

    Parameters
    ----------
    latitude : float
        Latitude of the destination.

    longitude : float
        Longitude of the destination.

    categories : list
        Geoapify categories to search.

    radius : int
        Search radius in meters.

    limit : int
        Maximum number of results requested per category.

    Returns
    -------
    list
        Cleaned place records containing the category used
        for the API request.
    """

    url = "https://api.geoapify.com/v2/places"

    all_places = []

    for category in categories:

        params = {
            "categories": category,
            "filter": f"circle:{longitude},{latitude},{radius}",
            "limit": limit,
            "apiKey": GEOAPIFY_API_KEY
        }

        try:
            response = requests.get(
                url,
                params=params,
                timeout=10
            )

            response.raise_for_status()

            data = response.json()

            for feature in data.get("features", []):

                properties = feature.get("properties", {})
                geometry = feature.get("geometry", {})

                coordinates = geometry.get(
                    "coordinates",
                    [None, None]
                )

                all_places.append({
                    "query_category": category,
                    "name": properties.get("name"),
                    "country": properties.get("country"),
                    "state": properties.get("state"),
                    "city": properties.get("city"),
                    "latitude": coordinates[1],
                    "longitude": coordinates[0],
                    "categories": properties.get(
                        "categories",
                        []
                    ),
                    "formatted_address": properties.get(
                        "formatted"
                    ),
                    "place_id": properties.get(
                        "place_id"
                    )
                })

        except requests.exceptions.RequestException as e:

            print(
                f"Failed category "
                f"{category}: {e}"
            )

    return all_places

In [ ]:
# Categories selected after testing the actual Geoapify responses.
# These categories cover attractions, culture, nature, recreation,
# and food availability.

places_categories = [
    "tourism.attraction",
    "tourism.sights",
    "leisure.park",
    "natural",
    "catering.restaurant"
]

print("Selected categories:")
for category in places_categories:
    print("-", category)

Selected categories:
- tourism.attraction
- tourism.sights
- leisure.park
- natural
- catering.restaurant


In [ ]:
# Collect multiple types of places for Goa.
# This is still a test; we are not collecting all 50 destinations yet.

goa_all_places = get_places_by_categories(
    latitude=goa_latitude,
    longitude=goa_longitude,
    categories=places_categories,
    radius=20000,
    limit=100
)

print(
    "Total raw records returned:",
    len(goa_all_places)
)

Total raw records returned: 296


In [ ]:
# Convert the collected places into a DataFrame.
goa_all_places_df = pd.DataFrame(goa_all_places)

# Count records before deduplication.
before_dedup = len(goa_all_places_df)

# Remove duplicate physical places using the unique Geoapify place_id.
# Keep the first occurrence of each place.
goa_unique_places_df = (
    goa_all_places_df
    .drop_duplicates(subset="place_id")
    .reset_index(drop=True)
)

after_dedup = len(goa_unique_places_df)

print("Records before deduplication:", before_dedup)
print("Unique places:", after_dedup)
print(
    "Duplicates removed:",
    before_dedup - after_dedup
)

Records before deduplication: 296
Unique places: 296
Duplicates removed: 0


In [ ]:
# Check how many records were returned for each API category.
# A count equal to our limit (100) is a warning that the category
# may contain more places than we collected.

category_counts = (
    goa_all_places_df
    .groupby("query_category")
    .size()
    .reset_index(name="records_returned")
    .sort_values("records_returned", ascending=False)
)

category_counts

,query_category,records_returned
0,catering.restaurant,100
2,natural,100
3,tourism.attraction,59
1,leisure.park,28
4,tourism.sights,9


In [ ]:
# Test the restaurant category again with a deliberately small limit.
# We are inspecting the response structure, not collecting final data.

restaurant_test = test_place_category(
    latitude=goa_latitude,
    longitude=goa_longitude,
    category="catering.restaurant",
    limit=10
)

print("Returned records:", len(restaurant_test))

Returned records: 10


In [ ]:
# Inspect the complete Geoapify response for metadata that may
# indicate pagination or additional available results.

url = "https://api.geoapify.com/v2/places"

params = {
    "categories": "catering.restaurant",
    "filter": f"circle:{goa_longitude},{goa_latitude},20000",
    "limit": 10,
    "apiKey": GEOAPIFY_API_KEY
}

response = requests.get(
    url,
    params=params,
    timeout=10
)

restaurant_response = response.json()

print(restaurant_response.keys())

dict_keys(['type', 'features'])


In [ ]:
# Inspect the top-level response metadata.
# This helps us determine whether Geoapify provides pagination
# information that we can use for complete collection.

for key, value in restaurant_response.items():

    if key != "features":
        print(f"\n{key}:")
        print(value)


type:
FeatureCollection


In [ ]:
# Compare the number of places returned at different radii.
# This helps us understand whether the 100-result ceiling is
# caused by a very large search area.

radii = [5000, 10000, 20000]

radius_results = []

for radius in radii:

    for category in [
        "catering.restaurant",
        "natural"
    ]:

        features = test_place_category(
            latitude=goa_latitude,
            longitude=goa_longitude,
            category=category,
            radius=radius,
            limit=100
        )

        radius_results.append({
            "radius_km": radius / 1000,
            "category": category,
            "places_returned": len(features),
            "limit_reached": len(features) == 100
        })

radius_test_df = pd.DataFrame(radius_results)

radius_test_df

,radius_km,category,places_returned,limit_reached
0,5.0,catering.restaurant,6,False
1,5.0,natural,80,False
2,10.0,catering.restaurant,39,False
3,10.0,natural,100,True
4,20.0,catering.restaurant,100,True
5,20.0,natural,100,True


In [ ]:
# Inspect the specific subcategories returned by Geoapify
# for natural places around Goa.

natural_features = test_place_category(
    latitude=goa_latitude,
    longitude=goa_longitude,
    category="natural",
    radius=5000,
    limit=100
)

natural_category_counter = Counter()

for feature in natural_features:

    properties = feature.get("properties", {})

    for category in properties.get("categories", []):
        natural_category_counter[category] += 1

natural_category_df = pd.DataFrame(
    natural_category_counter.items(),
    columns=["category", "count"]
).sort_values(
    "count",
    ascending=False
)

natural_category_df

,category,count
0,natural,80
5,natural.wetland,38
3,natural.forest,23
1,natural.water,14
6,natural.mountain,5
4,natural.water.river_system,5
7,natural.mountain.peak,5
2,natural.water.inland,2


In [ ]:
# Inspect the specific subcategories returned under tourism.attraction.
# This helps us decide which attraction features are meaningful.

attraction_features = test_place_category(
    latitude=goa_latitude,
    longitude=goa_longitude,
    category="tourism.attraction",
    radius=5000,
    limit=100
)

attraction_category_counter = Counter()

for feature in attraction_features:

    properties = feature.get("properties", {})

    for category in properties.get("categories", []):
        attraction_category_counter[category] += 1

attraction_category_df = pd.DataFrame(
    attraction_category_counter.items(),
    columns=["category", "count"]
).sort_values(
    "count",
    ascending=False
)

attraction_category_df

,category,count
0,tourism,1
1,tourism.attraction,1


In [ ]:
# ---------------------------------------------------------
# Places API configuration
# ---------------------------------------------------------
# These settings define how we will collect geographic
# and activity-related information for every destination.
# Keeping them in one place makes the pipeline easier to
# reproduce and modify later.
# ---------------------------------------------------------

PLACES_RADIUS_METERS = 5000
PLACES_LIMIT = 100

# Categories that produced meaningful information during
# our API exploration.
PLACES_CATEGORIES = [
    "tourism.sights",
    "leisure.park",
    "natural",
    "catering.restaurant"
]

print("Places radius:", PLACES_RADIUS_METERS, "meters")
print("Places limit:", PLACES_LIMIT)
print("Categories:", PLACES_CATEGORIES)

Places radius: 5000 meters
Places limit: 100
Categories: ['tourism.sights', 'leisure.park', 'natural', 'catering.restaurant']


In [ ]:
def collect_places(
    latitude,
    longitude,
    categories,
    radius=5000,
    limit=100
):
    """
    Collect raw place records from Geoapify for multiple categories.

    The function preserves:
    - The category used for the query
    - The returned place information
    - The original Geoapify categories
    - Whether the API result reached the requested limit

    Parameters
    ----------
    latitude : float
        Destination latitude.

    longitude : float
        Destination longitude.

    categories : list
        Geoapify categories to query.

    radius : int
        Search radius in meters.

    limit : int
        Maximum number of records requested per category.

    Returns
    -------
    list
        Raw place records collected from Geoapify.
    """

    url = "https://api.geoapify.com/v2/places"

    collected_places = []

    for category in categories:

        # Build parameters for this category.
        params = {
            "categories": category,
            "filter": f"circle:{longitude},{latitude},{radius}",
            "limit": limit,
            "apiKey": GEOAPIFY_API_KEY
        }

        try:
            # Request places from Geoapify.
            response = requests.get(
                url,
                params=params,
                timeout=10
            )

            response.raise_for_status()

            data = response.json()

            features = data.get("features", [])

            # If the number returned equals the API limit,
            # we flag it because there may be additional places.
            limit_reached = len(features) >= limit

            for feature in features:

                properties = feature.get(
                    "properties",
                    {}
                )

                geometry = feature.get(
                    "geometry",
                    {}
                )

                coordinates = geometry.get(
                    "coordinates",
                    [None, None]
                )

                collected_places.append({
                    "query_category": category,
                    "name": properties.get("name"),
                    "country": properties.get("country"),
                    "state": properties.get("state"),
                    "city": properties.get("city"),
                    "latitude": coordinates[1],
                    "longitude": coordinates[0],
                    "categories": properties.get(
                        "categories",
                        []
                    ),
                    "formatted_address": properties.get(
                        "formatted"
                    ),
                    "place_id": properties.get(
                        "place_id"
                    ),
                    "limit_reached": limit_reached
                })

            print(
                f"{category}: "
                f"{len(features)} places"
                + (
                    " [LIMIT REACHED]"
                    if limit_reached
                    else ""
                )
            )

        except requests.exceptions.RequestException as e:

            print(
                f"{category}: API request failed → {e}"
            )

    return collected_places

In [ ]:
# Test the final collection function on Goa before
# running it for all 50 destinations.

goa_places_final = collect_places(
    latitude=goa_latitude,
    longitude=goa_longitude,
    categories=PLACES_CATEGORIES,
    radius=PLACES_RADIUS_METERS,
    limit=PLACES_LIMIT
)

print(
    "\nTotal raw records collected:",
    len(goa_places_final)
)

tourism.sights: 0 places
leisure.park: 0 places
natural: 80 places
catering.restaurant: 6 places

Total raw records collected: 86


In [ ]:
# Convert Goa's raw Places records into a DataFrame.
goa_places_final_df = pd.DataFrame(goa_places_final)

# Show how many records were collected per category
# and whether that category hit the API limit.

goa_collection_summary = (
    goa_places_final_df
    .groupby("query_category")
    .agg(
        records=("place_id", "count"),
        limit_reached=("limit_reached", "max")
    )
    .reset_index()
)

goa_collection_summary

,query_category,records,limit_reached
0,catering.restaurant,6,False
1,natural,80,False


In [ ]:
# ---------------------------------------------------------
# Debug the two categories that unexpectedly returned 0.
# We compare their direct API responses using exactly the
# same radius and coordinates as our successful earlier test.
# ---------------------------------------------------------

debug_categories = [
    "tourism.sights",
    "leisure.park"
]

for category in debug_categories:

    url = "https://api.geoapify.com/v2/places"

    params = {
        "categories": category,
        "filter": f"circle:{goa_longitude},{goa_latitude},5000",
        "limit": 100,
        "apiKey": GEOAPIFY_API_KEY
    }

    response = requests.get(
        url,
        params=params,
        timeout=10
    )

    print("\n" + "=" * 60)
    print("CATEGORY:", category)
    print("STATUS:", response.status_code)

    data = response.json()

    print("TOP-LEVEL KEYS:", data.keys())
    print("FEATURE COUNT:", len(data.get("features", [])))

    if data.get("features"):
        print(
            "FIRST RESULT:",
            data["features"][0]["properties"]
        )


CATEGORY: tourism.sights
STATUS: 200
TOP-LEVEL KEYS: dict_keys(['type', 'features'])
FEATURE COUNT: 0

CATEGORY: leisure.park
STATUS: 200
TOP-LEVEL KEYS: dict_keys(['type', 'features'])
FEATURE COUNT: 0


In [ ]:
# ---------------------------------------------------------
# Places API configuration
# ---------------------------------------------------------
# Different place types need different search radii.
#
# Wider radius:
#   - sightseeing
#   - parks
#
# Smaller radius:
#   - natural features
#   - restaurants
#
# This prevents broad categories from immediately hitting
# the API's result limit while still capturing useful
# attractions around the destination.
# ---------------------------------------------------------

PLACES_CONFIG = {
    "tourism.sights": {
        "radius": 20000,
        "limit": 100
    },

    "leisure.park": {
        "radius": 20000,
        "limit": 100
    },

    "natural": {
        "radius": 5000,
        "limit": 100
    },

    "catering.restaurant": {
        "radius": 5000,
        "limit": 100
    }
}

print("Places API configuration:")

for category, config in PLACES_CONFIG.items():
    print(
        f"{category}: "
        f"{config['radius'] / 1000} km, "
        f"limit={config['limit']}"
    )

Places API configuration:
tourism.sights: 20.0 km, limit=100
leisure.park: 20.0 km, limit=100
natural: 5.0 km, limit=100
catering.restaurant: 5.0 km, limit=100


In [ ]:
def collect_places(
    latitude,
    longitude,
    places_config
):
    """
    Collect raw place records from Geoapify using
    category-specific search radii.

    Parameters
    ----------
    latitude : float
        Destination latitude.

    longitude : float
        Destination longitude.

    places_config : dict
        Dictionary containing category-specific radius
        and result-limit settings.

    Returns
    -------
    list
        Raw place records collected from Geoapify.
    """

    url = "https://api.geoapify.com/v2/places"

    collected_places = []

    # Process each category independently.
    for category, config in places_config.items():

        radius = config["radius"]
        limit = config["limit"]

        params = {
            "categories": category,
            "filter": f"circle:{longitude},{latitude},{radius}",
            "limit": limit,
            "apiKey": GEOAPIFY_API_KEY
        }

        try:

            response = requests.get(
                url,
                params=params,
                timeout=10
            )

            response.raise_for_status()

            data = response.json()

            features = data.get(
                "features",
                []
            )

            # If we receive exactly the requested limit,
            # flag the category because additional records
            # may exist outside the returned result set.
            limit_reached = len(features) >= limit

            print(
                f"{category}: "
                f"{len(features)} places | "
                f"radius={radius / 1000} km"
                + (
                    " | LIMIT REACHED"
                    if limit_reached
                    else ""
                )
            )

            # Extract each returned place.
            for feature in features:

                properties = feature.get(
                    "properties",
                    {}
                )

                geometry = feature.get(
                    "geometry",
                    {}
                )

                coordinates = geometry.get(
                    "coordinates",
                    [None, None]
                )

                collected_places.append({
                    "query_category": category,
                    "search_radius_km": radius / 1000,
                    "name": properties.get("name"),
                    "country": properties.get("country"),
                    "state": properties.get("state"),
                    "city": properties.get("city"),
                    "latitude": coordinates[1],
                    "longitude": coordinates[0],
                    "categories": properties.get(
                        "categories",
                        []
                    ),
                    "formatted_address": properties.get(
                        "formatted"
                    ),
                    "place_id": properties.get(
                        "place_id"
                    ),
                    "limit_reached": limit_reached
                })

        except requests.exceptions.RequestException as e:

            print(
                f"{category}: API request failed → {e}"
            )

    return collected_places

In [ ]:
# Test the category-specific collector using Goa.

goa_places_final = collect_places(
    latitude=goa_latitude,
    longitude=goa_longitude,
    places_config=PLACES_CONFIG
)

print(
    "\nTotal raw records collected:",
    len(goa_places_final)
)

tourism.sights: 9 places | radius=20.0 km
leisure.park: 28 places | radius=20.0 km
natural: 80 places | radius=5.0 km
catering.restaurant: 6 places | radius=5.0 km

Total raw records collected: 123


In [ ]:
# Convert the raw results into a DataFrame.

goa_places_final_df = pd.DataFrame(
    goa_places_final
)

goa_collection_summary = (
    goa_places_final_df
    .groupby("query_category")
    .agg(
        records=("place_id", "count"),
        radius_km=("search_radius_km", "first"),
        limit_reached=("limit_reached", "max")
    )
    .reset_index()
)

goa_collection_summary

,query_category,records,radius_km,limit_reached
0,catering.restaurant,6,5.0,False
1,leisure.park,28,20.0,False
2,natural,80,5.0,False
3,tourism.sights,9,20.0,False


In [ ]:
# ---------------------------------------------------------
# Collect Places data for all 50 destinations.
# ---------------------------------------------------------
# We use the validated latitude/longitude from
# destination_locations.csv and the category-specific
# radius configuration defined earlier.
#
# The raw API observations are preserved so that we can
# perform feature engineering later without repeatedly
# calling the API.
# ---------------------------------------------------------

all_places = []

total_destinations = len(validated_location_df)

for index, row in validated_location_df.iterrows():

    destination_id = row["destination_id"]
    destination = row["destination"]

    latitude = row["latitude"]
    longitude = row["longitude"]

    print(
        f"\n[{index + 1}/{total_destinations}] "
        f"Collecting places for: {destination}"
    )

    # Collect places for this destination.
    destination_places = collect_places(
        latitude=latitude,
        longitude=longitude,
        places_config=PLACES_CONFIG
    )

    # Add the destination identity to every API record.
    for place in destination_places:

        place["destination_id"] = destination_id
        place["destination"] = destination

    # Add this destination's records to the master list.
    all_places.extend(destination_places)

    print(
        f"Total records for {destination}: "
        f"{len(destination_places)}"
    )

print(
    f"\nFinished collection."
    f"\nTotal raw place records: {len(all_places)}"
)


[1/50] Collecting places for: Goa
tourism.sights: 9 places | radius=20.0 km
leisure.park: 28 places | radius=20.0 km
natural: 80 places | radius=5.0 km
catering.restaurant: 6 places | radius=5.0 km
Total records for Goa: 123

[2/50] Collecting places for: Munnar
tourism.sights: 2 places | radius=20.0 km
leisure.park: 19 places | radius=20.0 km
natural: 100 places | radius=5.0 km | LIMIT REACHED
catering.restaurant: 28 places | radius=5.0 km
Total records for Munnar: 149

[3/50] Collecting places for: Manali
tourism.sights: 3 places | radius=20.0 km
leisure.park: 4 places | radius=20.0 km
natural: 100 places | radius=5.0 km | LIMIT REACHED
catering.restaurant: 89 places | radius=5.0 km
Total records for Manali: 196

[4/50] Collecting places for: Jaipur
tourism.sights: 11 places | radius=20.0 km
leisure.park: 100 places | radius=20.0 km | LIMIT REACHED
natural: 39 places | radius=5.0 km
catering.restaurant: 100 places | radius=5.0 km | LIMIT REACHED
Total records for Jaipur: 250

[5/50]

In [ ]:
# Convert all raw API observations into one DataFrame.
places_raw_df = pd.DataFrame(all_places)

print("Rows:", len(places_raw_df))
print("Columns:", places_raw_df.columns.tolist())

Rows: 9448
Columns: ['query_category', 'search_radius_km', 'name', 'country', 'state', 'city', 'latitude', 'longitude', 'categories', 'formatted_address', 'place_id', 'limit_reached', 'destination_id', 'destination']


In [ ]:
places_raw_df.head()

,query_category,search_radius_km,name,country,state,city,latitude,longitude,categories,formatted_address,place_id,limit_reached,destination_id,destination
0,tourism.sights,20.0,Shri. Santadurga Kunkalikaran Prasann,India,Goa,Cuncolim,15.181434,74.000957,"[tourism, tourism.sights, tourism.sights.archa...","Shri. Santadurga Kunkalikaran Prasann, NH66, C...",51288d87ad0f80524059d8c8bfe0e45c2e40f00103f901...,False,1,Goa
1,tourism.sights,20.0,Palacio Do Deao,India,Goa,Quepem,15.213866,74.073557,"[building, building.historic, tourism, tourism...","Palacio Do Deao, MDR40, Quepem - 403705, Goa, ...",51dfb8d628b584524059947abbca7f6d2e40f00102f901...,False,1,Goa
2,tourism.sights,20.0,Memorial For Goan Victims of 1918 Spanish Flu,India,Goa,Margao,15.312981,73.983962,"[tourism, tourism.sights, tourism.sights.memor...","Memorial For Goan Victims of 1918 Spanish Flu,...",51dae0e93af97e5240599e9042fe3ea02e40f00103f901...,False,1,Goa
3,tourism.sights,20.0,Rivona Caves,India,Goa,Rivona,15.164889,74.111171,"[tourism, tourism.sights, tourism.sights.archa...","Rivona Caves, MDR36, Rivona - 403705, Goa, India",514c6a316d1d87524059ea3110476c542e40f00102f901...,False,1,Goa
4,tourism.sights,20.0,Cave at Ishwarbhat,India,Goa,Usgao,15.443551,74.040844,"[tourism, tourism.sights, tourism.sights.archa...","Cave at Ishwarbhat, MDR25, Bhamalwada, Usgao -...",5191c8412f9d825240594758df2919e32e40f00102f901...,False,1,Goa


In [ ]:
# ---------------------------------------------------------
# Basic quality checks on the complete raw Places dataset.
# ---------------------------------------------------------

print("Total records:", len(places_raw_df))

print(
    "Unique destinations:",
    places_raw_df["destination"].nunique()
)

print(
    "Unique place IDs:",
    places_raw_df["place_id"].nunique()
)

print(
    "Missing place IDs:",
    places_raw_df["place_id"].isna().sum()
)

print(
    "Missing names:",
    places_raw_df["name"].isna().sum()
)

print(
    "Duplicate place IDs:",
    places_raw_df["place_id"].duplicated().sum()
)

Total records: 9448
Unique destinations: 50
Unique place IDs: 9445
Missing place IDs: 0
Missing names: 3936
Duplicate place IDs: 3


In [ ]:
# ---------------------------------------------------------
# Identify destination/category combinations where the API
# returned the maximum requested number of records.
#
# These combinations may have additional places that were
# not returned by Geoapify.
# ---------------------------------------------------------

limit_warnings = (
    places_raw_df[
        places_raw_df["limit_reached"] == True
    ]
    .groupby(
        ["destination", "query_category"]
    )
    .size()
    .reset_index(
        name="records"
    )
)

limit_warnings

,destination,query_category,records
0,Agra,catering.restaurant,100
1,Agra,leisure.park,100
2,Ahmedabad,leisure.park,100
3,Alappuzha,natural,100
4,Amritsar,leisure.park,100
5,Bengaluru,catering.restaurant,100
6,Bengaluru,leisure.park,100
7,Bengaluru,natural,100
8,Bhopal,leisure.park,100
9,Bhubaneswar,leisure.park,100


In [ ]:
# ---------------------------------------------------------
# Save the complete raw Places API dataset.
# ---------------------------------------------------------
# We save the raw observations before doing any cleaning or
# feature engineering. This allows us to reproduce or change
# our feature engineering later without calling the API again.
# ---------------------------------------------------------

places_output_path = Path(
    "../data/raw/places/places_raw.csv"
)

# Create the directory if it does not already exist.
places_output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

# Save the complete raw dataset.
places_raw_df.to_csv(
    places_output_path,
    index=False
)

print(
    f"Raw Places dataset saved successfully:\n"
    f"{places_output_path}"
)

Raw Places dataset saved successfully:
..\data\raw\places\places_raw.csv


In [ ]:
# ---------------------------------------------------------
# Inspect duplicate place IDs
# ---------------------------------------------------------
# We first investigate the three duplicate physical places.
# A place may legitimately appear under more than one API
# category, so we need to understand the duplicates before
# deciding whether to remove them.
# ---------------------------------------------------------

duplicate_places = places_raw_df[
    places_raw_df["place_id"].duplicated(
        keep=False
    )
].sort_values("place_id")

print(
    "Duplicate records:",
    len(duplicate_places)
)

duplicate_places[
    [
        "destination",
        "query_category",
        "name",
        "place_id",
        "search_radius_km"
    ]
]

Duplicate records: 6


,destination,query_category,name,place_id,search_radius_km
2910,Alappuzha,natural,Vembanad Lake,510eb530c23a195340596f5cc69d99302340f00101f901...,5.0
3207,Kochi,natural,Vembanad Lake,510eb530c23a195340596f5cc69d99302340f00101f901...,5.0
3513,Thiruvananthapuram,leisure.park,Nissan Technopark,51233ec10f98365340597760a527d1372140f00103f901...,20.0
3782,Varkala,leisure.park,Nissan Technopark,51233ec10f98365340597760a527d1372140f00103f901...,20.0
3431,Thiruvananthapuram,tourism.sights,Kumaranasan Memorial,5139a5e61a2136534059080e301099442140f00102f901...,20.0
3753,Varkala,tourism.sights,Kumaranasan Memorial,5139a5e61a2136534059080e301099442140f00102f901...,20.0


In [ ]:
# ---------------------------------------------------------
# Check for duplicate places within the SAME destination.
# ---------------------------------------------------------
# A place appearing for two different destinations can be
# legitimate because our search areas overlap.
#
# However, the same place appearing twice for the same
# destination should be removed.
# ---------------------------------------------------------

same_destination_duplicates = (
    places_raw_df[
        places_raw_df.duplicated(
            subset=["destination", "place_id"],
            keep=False
        )
    ]
    .sort_values(
        ["destination", "place_id"]
    )
)

print(
    "Duplicate records within the same destination:",
    len(same_destination_duplicates)
)

same_destination_duplicates[
    [
        "destination",
        "query_category",
        "name",
        "place_id"
    ]
]

Duplicate records within the same destination: 0


,destination,query_category,name,place_id


In [ ]:
# ---------------------------------------------------------
# Analyze unnamed Places records
# ---------------------------------------------------------
# We do not delete unnamed records immediately.
# First, we determine which API categories contain them.
# This tells us whether they are useful geographic features
# or records that should be excluded from recommendation
# feature calculations.
# ---------------------------------------------------------

unnamed_places = places_raw_df[
    places_raw_df["name"].isna()
].copy()

print(
    "Total unnamed records:",
    len(unnamed_places)
)

# Count unnamed records by API query category.
unnamed_by_category = (
    unnamed_places
    .groupby("query_category")
    .size()
    .reset_index(name="unnamed_records")
    .sort_values(
        "unnamed_records",
        ascending=False
    )
)

unnamed_by_category

Total unnamed records: 3936


,query_category,unnamed_records
2,natural,2632
1,leisure.park,1135
0,catering.restaurant,169


In [ ]:
# Calculate the percentage of unnamed records
# within each API category.

category_totals = (
    places_raw_df
    .groupby("query_category")
    .size()
    .reset_index(name="total_records")
)

unnamed_analysis = category_totals.merge(
    unnamed_by_category,
    on="query_category",
    how="left"
)

# Categories with no unnamed records get zero.
unnamed_analysis["unnamed_records"] = (
    unnamed_analysis["unnamed_records"]
    .fillna(0)
    .astype(int)
)

# Calculate the percentage.
unnamed_analysis["unnamed_percentage"] = (
    unnamed_analysis["unnamed_records"]
    / unnamed_analysis["total_records"]
    * 100
)

unnamed_analysis

,query_category,total_records,unnamed_records,unnamed_percentage
0,catering.restaurant,2945,169,5.738540
1,leisure.park,2469,1135,45.970028
2,natural,3156,2632,83.396705
3,tourism.sights,878,0,0.000000


In [ ]:
# Inspect the category combinations associated with unnamed places.
# This helps us understand what these records actually represent.

unnamed_category_counter = Counter()

for categories in unnamed_places["categories"]:

    for category in categories:
        unnamed_category_counter[category] += 1

unnamed_category_df = pd.DataFrame(
    unnamed_category_counter.items(),
    columns=["category", "count"]
).sort_values(
    "count",
    ascending=False
)

unnamed_category_df.head(30)

,category,count
4,natural,2632
6,natural.water,1192
0,leisure,1136
1,leisure.park,1135
5,natural.forest,1048
11,natural.water.inland,514
8,natural.wetland,238
14,catering,169
15,catering.restaurant,169
7,natural.water.river_system,161


In [ ]:
# ---------------------------------------------------------
# Clean the raw Places dataset
# ---------------------------------------------------------
# IMPORTANT:
# We do not modify places_raw_df.
#
# The raw API dataset remains our original source of truth.
# We create a separate cleaned dataframe for feature
# engineering.
#
# Cleaning rules:
#
# 1. Remove exact duplicate observations within the same
#    destination + place_id.
#
# 2. Keep unnamed natural features because their geographic
#    category is still useful.
#
# 3. Require a name for restaurants.
#
# 4. Require a name for parks.
#
# 5. Keep tourism.sights records.
# ---------------------------------------------------------

places_clean_df = places_raw_df.copy()

# ---------------------------------------------------------
# Rule 1: Remove duplicate observations within the same
# destination.
#
# We already verified that there are currently zero such
# duplicates, but keeping this rule makes the pipeline robust.
# ---------------------------------------------------------

places_clean_df = (
    places_clean_df
    .drop_duplicates(
        subset=["destination", "place_id"]
    )
    .reset_index(drop=True)
)

print(
    "Rows after destination-level deduplication:",
    len(places_clean_df)
)

Rows after destination-level deduplication: 9448


In [ ]:
# ---------------------------------------------------------
# Apply category-specific quality rules.
# ---------------------------------------------------------

# Natural features:
# Keep both named and unnamed records because their specific
# geographic categories can still provide useful information.
natural_mask = (
    places_clean_df["query_category"] == "natural"
)

# Tourism sights:
# Keep all returned records because the category itself is
# meaningful and the current dataset has no unnamed sights.
sights_mask = (
    places_clean_df["query_category"] == "tourism.sights"
)

# Parks:
# Require a name because a large proportion of park records
# are unnamed.
park_mask = (
    (places_clean_df["query_category"] == "leisure.park")
    &
    (places_clean_df["name"].notna())
)

# Restaurants:
# Require a name because unnamed restaurant records are less
# reliable for destination-level food availability features.
restaurant_mask = (
    (places_clean_df["query_category"] == "catering.restaurant")
    &
    (places_clean_df["name"].notna())
)

# Keep records satisfying any of the category-specific rules.
places_clean_df = places_clean_df[
    natural_mask
    | sights_mask
    | park_mask
    | restaurant_mask
].reset_index(drop=True)

print(
    "Rows after category-specific cleaning:",
    len(places_clean_df)
)

Rows after category-specific cleaning: 8144


In [ ]:
# ---------------------------------------------------------
# Compare the raw and cleaned datasets.
# ---------------------------------------------------------

print(
    "Raw records:",
    len(places_raw_df)
)

print(
    "Cleaned records:",
    len(places_clean_df)
)

print(
    "Records removed:",
    len(places_raw_df) - len(places_clean_df)
)

Raw records: 9448
Cleaned records: 8144
Records removed: 1304


In [ ]:
# Check the cleaned dataset by category.

cleaned_category_summary = (
    places_clean_df
    .groupby("query_category")
    .agg(
        records=("place_id", "count"),
        unnamed_names=(
            "name",
            lambda x: x.isna().sum()
        )
    )
    .reset_index()
)

cleaned_category_summary

,query_category,records,unnamed_names
0,catering.restaurant,2776,0
1,leisure.park,1334,0
2,natural,3156,2632
3,tourism.sights,878,0


In [ ]:
# ---------------------------------------------------------
# Save the cleaned Places dataset.
# ---------------------------------------------------------
# The raw dataset remains untouched in:
# data/raw/places/places_raw.csv
#
# The cleaned version will be used for feature engineering.
# ---------------------------------------------------------

cleaned_places_path = Path(
    "../data/cleaned/places_cleaned.csv"
)

cleaned_places_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

places_clean_df.to_csv(
    cleaned_places_path,
    index=False
)

print(
    f"Cleaned Places dataset saved to:\n"
    f"{cleaned_places_path}"
)

Cleaned Places dataset saved to:
..\data\cleaned\places_cleaned.csv


In [ ]:
# ---------------------------------------------------------
# Inspect natural feature categories
# ---------------------------------------------------------
# We want to see exactly which natural subcategories occur
# across all 50 destinations before deciding which ones
# become ML features.
# ---------------------------------------------------------

natural_places = places_clean_df[
    places_clean_df["query_category"] == "natural"
].copy()

natural_category_counter = Counter()

for categories in natural_places["categories"]:

    for category in categories:

        # Only keep categories that belong to the natural
        # hierarchy. This avoids unrelated categories such
        # as access or building.
        if category.startswith("natural"):
            natural_category_counter[category] += 1

natural_category_df = pd.DataFrame(
    natural_category_counter.items(),
    columns=["category", "count"]
).sort_values(
    "count",
    ascending=False
)

natural_category_df

,category,count
0,natural,3156
1,natural.water,1579
3,natural.forest,1088
2,natural.water.inland,810
5,natural.wetland,243
4,natural.water.river_system,195
6,natural.mountain,109
12,natural.sand,59
14,natural.coastal,47
7,natural.mountain.peak,45


In [ ]:
# Create a copy so the cleaned place-level dataset remains unchanged.
places_features_df = places_clean_df.copy()

# Count tourism sights for each destination.
sight_counts = (
    places_features_df[
        places_features_df["query_category"] == "tourism.sights"
    ]
    .groupby("destination")
    .size()
    .rename("sight_count")
)

# Count named parks for each destination.
park_counts = (
    places_features_df[
        places_features_df["query_category"] == "leisure.park"
    ]
    .groupby("destination")
    .size()
    .rename("park_count")
)

# Count named restaurants for each destination.
restaurant_counts = (
    places_features_df[
        places_features_df["query_category"] == "catering.restaurant"
    ]
    .groupby("destination")
    .size()
    .rename("restaurant_count")
)

print("Sight destinations:", len(sight_counts))
print("Park destinations:", len(park_counts))
print("Restaurant destinations:", len(restaurant_counts))

Sight destinations: 45
Park destinations: 44
Restaurant destinations: 46


In [ ]:
# Count specific natural features for each destination.
# We use the category hierarchy to create meaningful
# destination-level environmental features.

natural_features = places_features_df[
    places_features_df["query_category"] == "natural"
].copy()

natural_counts = pd.DataFrame(
    index=natural_features["destination"].unique()
)

# Count water-related features.
natural_counts["water_count"] = (
    natural_features["categories"]
    .apply(lambda categories: "natural.water" in categories)
    .groupby(natural_features["destination"])
    .sum()
)

# Count forests.
natural_counts["forest_count"] = (
    natural_features["categories"]
    .apply(lambda categories: "natural.forest" in categories)
    .groupby(natural_features["destination"])
    .sum()
)

# Count wetlands.
natural_counts["wetland_count"] = (
    natural_features["categories"]
    .apply(lambda categories: "natural.wetland" in categories)
    .groupby(natural_features["destination"])
    .sum()
)

# Count river systems.
natural_counts["river_count"] = (
    natural_features["categories"]
    .apply(lambda categories: "natural.water.river_system" in categories)
    .groupby(natural_features["destination"])
    .sum()
)

# Count mountains.
natural_counts["mountain_count"] = (
    natural_features["categories"]
    .apply(lambda categories: "natural.mountain" in categories)
    .groupby(natural_features["destination"])
    .sum()
)

# Count coastal features.
natural_counts["coastal_count"] = (
    natural_features["categories"]
    .apply(lambda categories: "natural.coastal" in categories)
    .groupby(natural_features["destination"])
    .sum()
)

# Count sandy natural features.
natural_counts["sand_count"] = (
    natural_features["categories"]
    .apply(lambda categories: "natural.sand" in categories)
    .groupby(natural_features["destination"])
    .sum()
)

# Count protected natural areas.
natural_counts["protected_area_count"] = (
    natural_features["categories"]
    .apply(lambda categories: "natural.protected_area" in categories)
    .groupby(natural_features["destination"])
    .sum()
)

natural_counts = natural_counts.reset_index()

natural_counts.head()

,index,water_count,forest_count,wetland_count,river_count,mountain_count,coastal_count,sand_count,protected_area_count
0,Goa,14,23,38,5,5,0,0,0
1,Munnar,89,10,0,0,1,0,0,0
2,Manali,9,69,0,5,21,0,0,2
3,Jaipur,27,5,0,8,5,0,0,2
4,Udaipur,24,34,1,2,1,0,0,1


In [ ]:
# Rename the destination column correctly.
natural_counts = natural_counts.rename(
    columns={"index": "destination"}
)

# Combine sight, park, and restaurant counts.
places_feature_df = pd.concat(
    [
        sight_counts,
        park_counts,
        restaurant_counts
    ],
    axis=1
).reset_index()

# Rename the destination column.
places_feature_df = places_feature_df.rename(
    columns={"index": "destination"}
)

# Add the natural feature counts.
places_feature_df = places_feature_df.merge(
    natural_counts,
    on="destination",
    how="outer"
)

# Missing counts mean that no matching places were returned,
# so they should be represented as zero rather than NaN.
count_columns = [
    "sight_count",
    "park_count",
    "restaurant_count",
    "water_count",
    "forest_count",
    "wetland_count",
    "river_count",
    "mountain_count",
    "coastal_count",
    "sand_count",
    "protected_area_count"
]

places_feature_df[count_columns] = (
    places_feature_df[count_columns]
    .fillna(0)
    .astype(int)
)

print("Rows:", len(places_feature_df))
print("Columns:", places_feature_df.columns.tolist())

places_feature_df.head(10)

Rows: 50
Columns: ['destination', 'sight_count', 'park_count', 'restaurant_count', 'water_count', 'forest_count', 'wetland_count', 'river_count', 'mountain_count', 'coastal_count', 'sand_count', 'protected_area_count']


,destination,sight_count,park_count,restaurant_count,water_count,forest_count,wetland_count,river_count,mountain_count,coastal_count,sand_count,protected_area_count
0,Agra,13,60,98,42,32,3,1,0,0,0,0
1,Ahmedabad,12,52,40,14,2,0,2,0,0,0,0
2,Alappuzha,17,11,70,74,2,23,9,0,1,0,0
3,Amritsar,10,14,56,41,9,1,0,0,0,0,0
4,Andaman,0,0,0,0,2,0,0,0,0,0,0
5,Bengaluru,97,39,99,62,38,0,16,0,0,0,0
6,Bhopal,7,22,21,21,2,2,0,1,0,0,2
7,Bhubaneswar,6,69,39,16,4,2,0,5,0,0,0
8,Chennai,62,43,96,74,8,9,9,0,9,0,0
9,Coorg,0,4,0,0,2,0,0,0,0,0,0


In [ ]:
# Check that exactly 50 unique destinations are present.
print("Rows:", len(places_feature_df))
print(
    "Unique destinations:",
    places_feature_df["destination"].nunique()
)

# Check for duplicate destination rows.
print(
    "Duplicate destinations:",
    places_feature_df["destination"].duplicated().sum()
)

# Check for missing values in the feature columns.
print(
    "\nMissing values:"
)
print(
    places_feature_df[count_columns].isna().sum()
)

# Compare the collected destinations against the master
# destination list to identify any missing destinations.
master_destinations = set(
    validated_location_df["destination"]
)

feature_destinations = set(
    places_feature_df["destination"]
)

missing_destinations = (
    master_destinations - feature_destinations
)

extra_destinations = (
    feature_destinations - master_destinations
)

print(
    "\nMissing destinations:",
    missing_destinations
)

print(
    "Unexpected destinations:",
    extra_destinations
)

Rows: 50
Unique destinations: 50
Duplicate destinations: 0

Missing values:
sight_count             0
park_count              0
restaurant_count        0
water_count             0
forest_count            0
wetland_count           0
river_count             0
mountain_count          0
coastal_count           0
sand_count              0
protected_area_count    0
dtype: int64

Missing destinations: set()
Unexpected destinations: set()


In [ ]:
# Save the destination-level Places features.
# This file will later be combined with Weather, location,
# and cost-related features.

places_features_path = Path(
    "../data/cleaned/places_features.csv"
)

places_features_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

places_feature_df.to_csv(
    places_features_path,
    index=False
)

print(
    f"Places feature dataset saved successfully:\n"
    f"{places_features_path}"
)

Places feature dataset saved successfully:
..\data\cleaned\places_features.csv


In [31]:
# Import the libraries needed for the next analysis.
import pandas as pd
from pathlib import Path

# Load the already-created Places feature dataset.
places_features_path = Path(
    "../data/cleaned/places_features.csv"
)

places_feature_df = pd.read_csv(
    places_features_path
)

# List the feature columns we created.
count_columns = [
    "sight_count",
    "park_count",
    "restaurant_count",
    "water_count",
    "forest_count",
    "wetland_count",
    "river_count",
    "mountain_count",
    "coastal_count",
    "sand_count",
    "protected_area_count"
]

print("Rows:", len(places_feature_df))
print("Columns:", places_feature_df.columns.tolist())

Rows: 50
Columns: ['destination', 'sight_count', 'park_count', 'restaurant_count', 'water_count', 'forest_count', 'wetland_count', 'river_count', 'mountain_count', 'coastal_count', 'sand_count', 'protected_area_count']


In [32]:
# Display descriptive statistics for all Places features.
# This helps us understand the range and distribution
# of each feature across the 50 destinations.

places_feature_df[count_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
sight_count,50.0,17.56,27.069947,0.0,3.00,7.5,12.75,100.0
park_count,50.0,26.68,24.537718,0.0,4.00,19.5,42.75,78.0
restaurant_count,50.0,55.52,36.703022,0.0,25.50,56.0,92.00,100.0
water_count,50.0,31.58,29.512426,0.0,9.00,20.5,45.00,98.0
forest_count,50.0,21.76,25.083176,0.0,3.25,11.0,31.75,97.0
wetland_count,50.0,4.86,9.952971,0.0,0.00,1.0,4.00,41.0
river_count,50.0,3.90,4.726175,0.0,0.00,2.0,6.75,17.0
mountain_count,50.0,2.18,3.804884,0.0,0.00,1.0,2.75,21.0
coastal_count,50.0,0.94,2.721119,0.0,0.00,0.0,0.00,12.0
sand_count,50.0,1.18,5.516913,0.0,0.00,0.0,0.00,34.0


In [33]:
# Show the minimum and maximum value for each feature.
# This makes it easy to identify features with very
# large ranges or suspiciously high values.

places_feature_df[count_columns].agg(
    ["min", "max"]
).T

,min,max
sight_count,0,100
park_count,0,78
restaurant_count,0,100
water_count,0,98
forest_count,0,97
wetland_count,0,41
river_count,0,17
mountain_count,0,21
coastal_count,0,12
sand_count,0,34


In [34]:
# Load the raw Places data so we can identify which
# destination/category combinations reached the API limit.

places_raw_df = pd.read_csv(
    "../data/raw/places/places_raw.csv"
)

# Count the number of destination/category combinations
# where Geoapify returned the maximum requested records.

limit_summary = (
    places_raw_df[
        places_raw_df["limit_reached"] == True
    ]
    .groupby("query_category")
    .agg(
        capped_destination_count=("destination", "nunique")
    )
    .reset_index()
)

limit_summary

,query_category,capped_destination_count
0,catering.restaurant,17
1,leisure.park,19
2,natural,18
3,tourism.sights,2


In [35]:
# Show exactly which destinations were affected by the
# API result limit for each category.

capped_destinations = (
    places_raw_df[
        places_raw_df["limit_reached"] == True
    ]
    .groupby("query_category")["destination"]
    .unique()
)

for category, destinations in capped_destinations.items():

    print("\nCategory:", category)
    print("Number of capped destinations:", len(destinations))
    print("Destinations:")

    for destination in sorted(destinations):
        print("-", destination)


Category: catering.restaurant
Number of capped destinations: 17
Destinations:
- Agra
- Bengaluru
- Chennai
- Delhi
- Dharamshala
- Jaipur
- Kochi
- Kolkata
- Mumbai
- Mysore
- Pondicherry
- Pune
- Rishikesh
- Srinagar
- Thiruvananthapuram
- Udaipur
- Varkala

Category: leisure.park
Number of capped destinations: 19
Destinations:
- Agra
- Ahmedabad
- Amritsar
- Bengaluru
- Bhopal
- Bhubaneswar
- Chennai
- Delhi
- Hyderabad
- Indore
- Jaipur
- Kochi
- Kolkata
- Mumbai
- Mysore
- Pune
- Srinagar
- Thiruvananthapuram
- Varanasi

Category: natural
Number of capped destinations: 18
Destinations:
- Alappuzha
- Bengaluru
- Chennai
- Darjeeling
- Delhi
- Gokarna
- Kaziranga
- Kochi
- Kolkata
- Manali
- Mumbai
- Munnar
- Pondicherry
- Pune
- Shimla
- Thiruvananthapuram
- Varkala
- Wayanad

Category: tourism.sights
Number of capped destinations: 2
Destinations:
- Delhi
- Hyderabad


In [36]:
# Create a summary showing how often each category
# reached the API result limit.

limit_check = (
    places_raw_df
    .groupby(["destination", "query_category"])
    .agg(
        records_returned=("place_id", "count"),
        limit_reached=("limit_reached", "max")
    )
    .reset_index()
)

# Show the categories that reached the limit.
limit_check[
    limit_check["limit_reached"] == True
].sort_values(
    ["query_category", "destination"]
)

,destination,query_category,records_returned,limit_reached
0,Agra,catering.restaurant,100,True
17,Bengaluru,catering.restaurant,100,True
29,Chennai,catering.restaurant,100,True
39,Delhi,catering.restaurant,100,True
43,Dharamshala,catering.restaurant,100,True
70,Jaipur,catering.restaurant,100,True
87,Kochi,catering.restaurant,100,True
95,Kolkata,catering.restaurant,100,True
109,Mumbai,catering.restaurant,100,True
121,Mysore,catering.restaurant,100,True


In [37]:
# Count how many destinations reached the API limit for
# each individual feature category.

capped_summary = (
    limit_check[
        limit_check["limit_reached"] == True
    ]
    .groupby("query_category")
    .agg(
        capped_destinations=("destination", "nunique")
    )
    .reset_index()
)

# Add the total number of destinations for comparison.
capped_summary["total_destinations"] = 50

# Calculate the percentage of destinations affected.
capped_summary["capped_percentage"] = (
    capped_summary["capped_destinations"]
    / capped_summary["total_destinations"]
    * 100
)

capped_summary

,query_category,capped_destinations,total_destinations,capped_percentage
0,catering.restaurant,17,50,34.0
1,leisure.park,19,50,38.0
2,natural,18,50,36.0
3,tourism.sights,2,50,4.0


In [38]:
# Show the destinations with the highest raw counts.
# This helps us understand which destinations are likely
# to be affected most strongly by the API ceiling.

places_feature_df.sort_values(
    "restaurant_count",
    ascending=False
)[[
    "destination",
    "restaurant_count"
]].head(15)

,destination,restaurant_count
11,Delhi,100
38,Pune,100
26,Kolkata,99
44,Srinagar,99
30,Mumbai,99
5,Bengaluru,99
12,Dharamshala,98
0,Agra,98
24,Kochi,98
8,Chennai,96


In [39]:
# Calculate percentile ranks for each Places feature.
# Percentile rank shows how a destination compares with
# the other 49 destinations without treating the raw count
# as an exact real-world total.

places_model_df = places_feature_df.copy()

for column in count_columns:
    places_model_df[f"{column}_score"] = (
        places_model_df[column]
        .rank(method="average", pct=True)
    )

places_model_df.head()

,destination,sight_count,park_count,restaurant_count,water_count,forest_count,wetland_count,river_count,mountain_count,coastal_count,...,park_count_score,restaurant_count_score,water_count_score,forest_count_score,wetland_count_score,river_count_score,mountain_count_score,coastal_count_score,sand_count_score,protected_area_count_score
0,Agra,13,60,98,42,32,3,1,0,0,...,0.86,0.86,0.70,0.76,0.70,0.44,0.22,0.43,0.44,0.3
1,Ahmedabad,12,52,40,14,2,0,2,0,0,...,0.82,0.38,0.31,0.19,0.24,0.50,0.22,0.43,0.44,0.3
2,Alappuzha,17,11,70,74,2,23,9,0,1,...,0.36,0.57,0.85,0.19,0.94,0.87,0.22,0.86,0.44,0.3
3,Amritsar,10,14,56,41,9,1,0,0,0,...,0.48,0.51,0.68,0.45,0.53,0.21,0.22,0.43,0.44,0.3
4,Andaman,0,0,0,0,2,0,0,0,0,...,0.07,0.05,0.06,0.19,0.24,0.21,0.22,0.43,0.44,0.3


In [40]:
# Compare capped and non-capped destinations for each
# major API category.

capped_comparison = (
    limit_check
    .merge(
        places_feature_df[
            [
                "destination",
                "sight_count",
                "park_count",
                "restaurant_count"
            ]
        ],
        on="destination",
        how="left"
    )
)

capped_comparison[
    capped_comparison["limit_reached"] == True
].head(20)

,destination,query_category,records_returned,limit_reached,sight_count,park_count,restaurant_count
0,Agra,catering.restaurant,100,True,13,60,98
1,Agra,leisure.park,100,True,13,60,98
5,Ahmedabad,leisure.park,100,True,12,52,40
10,Alappuzha,natural,100,True,17,11,70
13,Amritsar,leisure.park,100,True,10,14,56
17,Bengaluru,catering.restaurant,100,True,97,39,99
18,Bengaluru,leisure.park,100,True,97,39,99
19,Bengaluru,natural,100,True,97,39,99
22,Bhopal,leisure.park,100,True,7,22,21
26,Bhubaneswar,leisure.park,100,True,6,69,39


In [41]:
# Create a lookup showing whether each destination was capped
# for each individual API category.

capped_flags = (
    limit_check
    .pivot(
        index="destination",
        columns="query_category",
        values="limit_reached"
    )
    .fillna(False)
    .reset_index()
)

# Rename columns to make the meaning explicit.
capped_flags = capped_flags.rename(
    columns={
        "catering.restaurant": "restaurant_capped",
        "leisure.park": "park_capped",
        "natural": "natural_capped",
        "tourism.sights": "sights_capped"
    }
)

# Combine the cap information with our model dataframe.
places_model_df = places_model_df.merge(
    capped_flags,
    on="destination",
    how="left"
)

places_model_df.head()

,destination,sight_count,park_count,restaurant_count,water_count,forest_count,wetland_count,river_count,mountain_count,coastal_count,...,wetland_count_score,river_count_score,mountain_count_score,coastal_count_score,sand_count_score,protected_area_count_score,restaurant_capped,park_capped,natural_capped,sights_capped
0,Agra,13,60,98,42,32,3,1,0,0,...,0.70,0.44,0.22,0.43,0.44,0.3,True,True,False,False
1,Ahmedabad,12,52,40,14,2,0,2,0,0,...,0.24,0.50,0.22,0.43,0.44,0.3,False,True,False,False
2,Alappuzha,17,11,70,74,2,23,9,0,1,...,0.94,0.87,0.22,0.86,0.44,0.3,False,False,True,False
3,Amritsar,10,14,56,41,9,1,0,0,0,...,0.53,0.21,0.22,0.43,0.44,0.3,False,True,False,False
4,Andaman,0,0,0,0,2,0,0,0,0,...,0.24,0.21,0.22,0.43,0.44,0.3,False,False,False,False


In [42]:
# Compare restaurant availability scores for capped
# versus non-capped destinations.

places_model_df.groupby(
    "restaurant_capped"
)["restaurant_count_score"].agg(
    ["count", "mean", "min", "max"]
)

,count,mean,min,max
restaurant_capped,,,,
False,33,0.344545,0.05,0.70
True,17,0.831176,0.57,0.99


In [44]:
# Compare percentile scores between capped and non-capped
# destinations for every major Places category.

cap_score_comparison = []

category_score_pairs = {
    "restaurant_capped": "restaurant_count_score",
    "park_capped": "park_count_score",
    "natural_capped": "water_count_score",
    "sights_capped": "sight_count_score"
}

for cap_column, score_column in category_score_pairs.items():

    grouped = (
        places_model_df
        .groupby(cap_column)[score_column]
        .agg(["count", "mean", "min", "max"])
        .reset_index()
    )

    grouped["feature"] = score_column

    cap_score_comparison.append(grouped)

cap_score_comparison_df = pd.concat(
    cap_score_comparison,
    ignore_index=True
)

cap_score_comparison_df

,restaurant_capped,count,mean,min,max,feature,park_capped,natural_capped,sights_capped
0,False,33,0.344545,0.05,0.70,restaurant_count_score,NaN,NaN,NaN
1,True,17,0.831176,0.57,0.99,restaurant_count_score,NaN,NaN,NaN
2,NaN,31,0.333226,0.07,0.71,park_count_score,False,NaN,NaN
3,NaN,19,0.798421,0.48,1.00,park_count_score,True,NaN,NaN
4,NaN,32,0.408125,0.06,0.90,water_count_score,NaN,False,NaN
5,NaN,18,0.691111,0.06,0.99,water_count_score,NaN,True,NaN
6,NaN,48,0.490000,0.06,0.96,sight_count_score,NaN,NaN,False
7,NaN,2,0.990000,0.99,0.99,sight_count_score,NaN,NaN,True


In [45]:
# Create a separate dataframe for the recommendation model.
# The original Places feature dataset remains unchanged.

places_model_features = places_model_df[
    [
        "destination",
        "sight_count_score",
        "park_count_score",
        "restaurant_count_score",
        "water_count_score",
        "forest_count_score",
        "wetland_count_score",
        "river_count_score",
        "mountain_count_score",
        "coastal_count_score",
        "sand_count_score",
        "protected_area_count_score",
        "sights_capped",
        "park_capped",
        "restaurant_capped",
        "natural_capped"
    ]
].copy()

# Rename the score columns to cleaner model feature names.
places_model_features = places_model_features.rename(
    columns={
        "sight_count_score": "sight_score",
        "park_count_score": "park_score",
        "restaurant_count_score": "restaurant_score",
        "water_count_score": "water_score",
        "forest_count_score": "forest_score",
        "wetland_count_score": "wetland_score",
        "river_count_score": "river_score",
        "mountain_count_score": "mountain_score",
        "coastal_count_score": "coastal_score",
        "sand_count_score": "sand_score",
        "protected_area_count_score": "protected_area_score"
    }
)

places_model_features.head()

,destination,sight_score,park_score,restaurant_score,water_score,forest_score,wetland_score,river_score,mountain_score,coastal_score,sand_score,protected_area_score,sights_capped,park_capped,restaurant_capped,natural_capped
0,Agra,0.76,0.86,0.86,0.70,0.76,0.70,0.44,0.22,0.43,0.44,0.3,False,True,True,False
1,Ahmedabad,0.74,0.82,0.38,0.31,0.19,0.24,0.50,0.22,0.43,0.44,0.3,False,True,False,False
2,Alappuzha,0.78,0.36,0.57,0.85,0.19,0.94,0.87,0.22,0.86,0.44,0.3,False,False,False,True
3,Amritsar,0.66,0.48,0.51,0.68,0.45,0.53,0.21,0.22,0.43,0.44,0.3,False,True,False,False
4,Andaman,0.06,0.07,0.05,0.06,0.19,0.24,0.21,0.22,0.43,0.44,0.3,False,False,False,False


In [46]:
# Check the final Places model feature dataset.

print("Rows:", len(places_model_features))
print(
    "Unique destinations:",
    places_model_features["destination"].nunique()
)

print(
    "\nMissing values:"
)

print(
    places_model_features.isna().sum()
)

Rows: 50
Unique destinations: 50

Missing values:
destination             0
sight_score             0
park_score              0
restaurant_score        0
water_score             0
forest_score            0
wetland_score           0
river_score             0
mountain_score          0
coastal_score           0
sand_score              0
protected_area_score    0
sights_capped           0
park_capped             0
restaurant_capped       0
natural_capped          0
dtype: int64


In [47]:
# Save the final Places features used by the recommendation model.
# This file contains normalized scores and API-cap indicators
# for all 50 destinations.

places_model_features.to_csv(
    "../data/cleaned/places_model_features.csv",
    index=False
)

print("Places model features saved successfully.")

Places model features saved successfully.


In [48]:
# Reload the saved file to confirm that it was written correctly.

places_check = pd.read_csv(
    "../data/cleaned/places_model_features.csv"
)

print("Rows:", len(places_check))
print("Unique destinations:", places_check["destination"].nunique())
print("Columns:", places_check.columns.tolist())

Rows: 50
Unique destinations: 50
Columns: ['destination', 'sight_score', 'park_score', 'restaurant_score', 'water_score', 'forest_score', 'wetland_score', 'river_score', 'mountain_score', 'coastal_score', 'sand_score', 'protected_area_score', 'sights_capped', 'park_capped', 'restaurant_capped', 'natural_capped']


#### Combining the Processed Data Till now

In [50]:
# Check which cleaned datasets currently exist.
# This avoids rerunning any API collection code.

from pathlib import Path

cleaned_dir = Path("../data/cleaned")

print("Files currently available in data/cleaned:")
for file in sorted(cleaned_dir.iterdir()):
    print("-", file.name)

Files currently available in data/cleaned:
- places_cleaned.csv
- places_features.csv
- places_model_features.csv


In [51]:
# Load the final Places features that will be used by the recommendation model.
# This avoids rerunning the Geoapify API collection process.

places_model_features = pd.read_csv(
    "../data/cleaned/places_model_features.csv"
)

print("Rows:", len(places_model_features))
print("Unique destinations:", places_model_features["destination"].nunique())
print("Columns:", places_model_features.columns.tolist())

Rows: 50
Unique destinations: 50
Columns: ['destination', 'sight_score', 'park_score', 'restaurant_score', 'water_score', 'forest_score', 'wetland_score', 'river_score', 'mountain_score', 'coastal_score', 'sand_score', 'protected_area_score', 'sights_capped', 'park_capped', 'restaurant_capped', 'natural_capped']


In [52]:
# Check all datasets currently available in the project.
# This helps us identify the already-saved weather, location,
# hotel, flight, and destination datasets without rerunning APIs.

from pathlib import Path

for folder in ["../data/raw", "../data/cleaned"]:
    print(f"\nFiles in {folder}:")
    
    path = Path(folder)
    
    for file in sorted(path.rglob("*")):
        if file.is_file():
            print("-", file.relative_to(path))


Files in ../data/raw:
- destination_master.csv
- places\destination_locations.csv
- places\places_raw.csv
- weather\weather_test.csv

Files in ../data/cleaned:
- places_cleaned.csv
- places_features.csv
- places_model_features.csv


In [53]:
# Load the saved datasets from the previous API/data-processing steps.
# We use the saved files so that we do NOT need to call the APIs again.

destination_master = pd.read_csv(
    "../data/raw/destination_master.csv"
)

weather_data = pd.read_csv(
    "../data/raw/weather/weather_test.csv"
)

places_model_features = pd.read_csv(
    "../data/cleaned/places_model_features.csv"
)

print("Destination master:", destination_master.shape)
print("Weather data:", weather_data.shape)
print("Places features:", places_model_features.shape)

Destination master: (50, 3)
Weather data: (5, 15)
Places features: (50, 16)


In [54]:
# Inspect the columns and destination names in each saved dataset.
# We do this before merging to make sure the keys and coverage are correct.

print("DESTINATION MASTER")
print(destination_master.columns.tolist())
print(destination_master.head())

print("\nWEATHER DATA")
print(weather_data.columns.tolist())
print(weather_data[["destination"]].to_string(index=False))

print("\nPLACES FEATURES")
print(places_model_features.columns.tolist())
print(places_model_features[["destination"]].head())

DESTINATION MASTER
['destination_id', 'destination', 'search_query']
   destination_id destination search_query
0               1         Goa          Goa
1               2      Munnar       Munnar
2               3      Manali       Manali
3               4      Jaipur       Jaipur
4               5     Udaipur      Udaipur

WEATHER DATA
['destination', 'country', 'latitude', 'longitude', 'temperature', 'feels_like', 'humidity', 'pressure', 'wind_speed', 'cloudiness', 'weather_condition', 'weather_description', 'visibility', 'rain_1h', 'timestamp']
destination
  Hyderabad
        Goa
     Munnar
     Manali
     Jaipur

PLACES FEATURES
['destination', 'sight_score', 'park_score', 'restaurant_score', 'water_score', 'forest_score', 'wetland_score', 'river_score', 'mountain_score', 'coastal_score', 'sand_score', 'protected_area_score', 'sights_capped', 'park_capped', 'restaurant_capped', 'natural_capped']
  destination
0        Agra
1   Ahmedabad
2   Alappuzha
3    Amritsar
4     Andaman

In [55]:
# Find which destinations already have weather data
# and which destinations still need to be collected.

all_destinations = set(destination_master["destination"])

weather_destinations = set(weather_data["destination"])

missing_weather = sorted(
    all_destinations - weather_destinations
)

print("Total destinations:", len(all_destinations))
print("Weather already available:", len(weather_destinations))
print("Weather still missing:", len(missing_weather))

print("\nMissing weather destinations:")
print(missing_weather)

Total destinations: 50
Weather already available: 5
Weather still missing: 45

Missing weather destinations:
['Agra', 'Ahmedabad', 'Alappuzha', 'Amritsar', 'Andaman', 'Bengaluru', 'Bhopal', 'Bhubaneswar', 'Chennai', 'Coorg', 'Darjeeling', 'Delhi', 'Dharamshala', 'Gangtok', 'Gokarna', 'Hampi', 'Indore', 'Jaisalmer', 'Jim Corbett', 'Jodhpur', 'Kaziranga', 'Kochi', 'Kodaikanal', 'Kolkata', 'Ladakh', 'Mahabalipuram', 'Mumbai', 'Mussoorie', 'Mysore', 'Nainital', 'Ooty', 'Pahalgam', 'Pondicherry', 'Pune', 'Ranchi', 'Ranthambore', 'Rishikesh', 'Shillong', 'Shimla', 'Srinagar', 'Thiruvananthapuram', 'Udaipur', 'Varanasi', 'Varkala', 'Wayanad']


### collectiong weather data for remaining locations

In [57]:
# Collect weather data only for destinations that are missing.
# The existing 5 weather records will NOT be requested again,
# which saves API calls and avoids duplicate records.

new_weather_records = []

for destination in missing_weather:

    print(f"Collecting weather: {destination}")

    weather_result = get_weather(destination)

    # Only keep successful API responses.
    if weather_result is not None:
        new_weather_records.append(weather_result)

print("\nNew weather records collected:", len(new_weather_records))

API request failed for Coorg: 404 Client Error: Not Found for url: https://api.openweathermap.org/data/2.5/weather?q=Coorg&appid=05405a031eac323c1b7d020746c2e3e0&units=metric
API request failed for Dharamshala: 404 Client Error: Not Found for url: https://api.openweathermap.org/data/2.5/weather?q=Dharamshala&appid=05405a031eac323c1b7d020746c2e3e0&units=metric
API request failed for Jim Corbett: 404 Client Error: Not Found for url: https://api.openweathermap.org/data/2.5/weather?q=Jim+Corbett&appid=05405a031eac323c1b7d020746c2e3e0&units=metric
API request failed for Kaziranga: 404 Client Error: Not Found for url: https://api.openweathermap.org/data/2.5/weather?q=Kaziranga&appid=05405a031eac323c1b7d020746c2e3e0&units=metric
API request failed for Ladakh: 404 Client Error: Not Found for url: https://api.openweathermap.org/data/2.5/weather?q=Ladakh&appid=05405a031eac323c1b7d020746c2e3e0&units=metric
API request failed for Ranthambore: 404 Client Error: Not Found for url: https://api.openwe

In [58]:
# Convert the newly collected weather records into a DataFrame
# so they can be combined with the existing 5 records.

new_weather_data = pd.DataFrame(new_weather_records)

print("Rows:", len(new_weather_data))
print("Columns:", new_weather_data.columns.tolist())

Rows: 38
Columns: ['destination', 'country', 'latitude', 'longitude', 'temperature', 'feels_like', 'humidity', 'pressure', 'wind_speed', 'cloudiness', 'weather_condition', 'weather_description', 'visibility', 'rain_1h', 'timestamp']


In [59]:
# Destinations that failed with the simple city-name weather query.
# We will retry these using their known geographic coordinates.

failed_weather_destinations = [
    "Coorg",
    "Dharamshala",
    "Jim Corbett",
    "Kaziranga",
    "Ladakh",
    "Ranthambore",
    "Wayanad"
]

print("Destinations requiring coordinate-based weather lookup:")
print(failed_weather_destinations)

Destinations requiring coordinate-based weather lookup:
['Coorg', 'Dharamshala', 'Jim Corbett', 'Kaziranga', 'Ladakh', 'Ranthambore', 'Wayanad']


In [60]:
# Load the validated Geoapify location dataset.
# This contains the latitude and longitude for all 50 destinations.

destination_locations = pd.read_csv(
    "../data/raw/places/destination_locations.csv"
)

print("Rows:", len(destination_locations))
print(destination_locations.columns.tolist())

Rows: 50
['destination_id', 'destination', 'search_query', 'resolved_name', 'country', 'country_code', 'state', 'state_code', 'latitude', 'longitude', 'result_type', 'formatted_address', 'confidence', 'match_type', 'place_id']


In [61]:
# Check the coordinates for the seven destinations
# that failed during the weather API request.

failed_locations = destination_locations[
    destination_locations["destination"].isin(
        failed_weather_destinations
    )
]

failed_locations[
    ["destination", "latitude", "longitude"]
]

,destination,latitude,longitude
17,Coorg,12.382733,75.664072
18,Wayanad,11.715129,76.127118
32,Ladakh,33.945641,77.656858
34,Dharamshala,32.214304,76.319672
46,Kaziranga,26.589253,93.408474
47,Jim Corbett,29.557475,78.842495
48,Ranthambore,26.018892,76.352244


In [62]:
# Fetch weather using latitude and longitude.
# This is used for destinations that could not be found
# reliably using their destination names.


# Co-ordinate based weather function

import requests

def get_weather_by_coordinates(destination, latitude, longitude):
    """
    Fetch current weather data using geographic coordinates.
    """

    try:
        url = "https://api.openweathermap.org/data/2.5/weather"

        params = {
            "lat": latitude,
            "lon": longitude,
            "appid": WEATHER_API_KEY,
            "units": "metric"
        }

        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()

        data = response.json()

        # Extract the same fields used by our normal weather function.
        return {
            "destination": destination,
            "country": data.get("sys", {}).get("country"),
            "latitude": data.get("coord", {}).get("lat"),
            "longitude": data.get("coord", {}).get("lon"),
            "temperature": data.get("main", {}).get("temp"),
            "feels_like": data.get("main", {}).get("feels_like"),
            "humidity": data.get("main", {}).get("humidity"),
            "pressure": data.get("main", {}).get("pressure"),
            "wind_speed": data.get("wind", {}).get("speed"),
            "cloudiness": data.get("clouds", {}).get("all"),
            "weather_condition": (
                data.get("weather", [{}])[0].get("main")
            ),
            "weather_description": (
                data.get("weather", [{}])[0].get("description")
            ),
            "visibility": data.get("visibility"),
            "rain_1h": data.get("rain", {}).get("1h", 0.0),
            "timestamp": pd.Timestamp.now(tz="UTC").isoformat()
        }

    except Exception as e:
        print(f"Coordinate weather request failed for {destination}: {e}")
        return None

In [63]:
# Retry weather collection for the seven destinations
# using their validated latitude and longitude.

coordinate_weather_records = []

for _, row in failed_locations.iterrows():

    print(f"Collecting weather by coordinates: {row['destination']}")

    result = get_weather_by_coordinates(
        destination=row["destination"],
        latitude=row["latitude"],
        longitude=row["longitude"]
    )

    if result is not None:
        coordinate_weather_records.append(result)

print("\nCoordinate-based weather records collected:",
      len(coordinate_weather_records))


Coordinate-based weather records collected: 7


In [64]:
# Convert the successful coordinate-based responses into a DataFrame.

coordinate_weather_data = pd.DataFrame(
    coordinate_weather_records
)

print("Rows:", len(coordinate_weather_data))
print(
    coordinate_weather_data[
        ["destination", "latitude", "longitude", "temperature"]
    ]
)

Rows: 7
   destination  latitude  longitude  temperature
0        Coorg   12.3755    75.6625        19.00
1      Wayanad   11.7151    76.1271        20.58
2       Ladakh   33.9456    77.6569        15.11
3  Dharamshala   32.2143    76.3197        20.49
4    Kaziranga   26.5893    93.4085        25.46
5  Jim Corbett   29.5575    78.8425        26.25
6  Ranthambore   26.0189    76.3522        28.65


In [65]:
# Combine the existing 5 weather records,
# the 38 successfully collected records,
# and the 7 coordinate-based records.

weather_complete = pd.concat(
    [
        weather_data,
        new_weather_data,
        coordinate_weather_data
    ],
    ignore_index=True
)

print("Total weather records:", len(weather_complete))

Total weather records: 50


In [66]:
# Validate that we have exactly one weather record
# for each of our 50 destinations.

print("Rows:", len(weather_complete))
print("Unique destinations:", weather_complete["destination"].nunique())

print("\nDuplicate destinations:")
print(
    weather_complete[
        weather_complete["destination"].duplicated(keep=False)
    ]["destination"].unique()
)

print("\nMissing values:")
print(weather_complete.isnull().sum())

Rows: 50
Unique destinations: 50

Duplicate destinations:
<StringArray>
[]
Length: 0, dtype: str

Missing values:
destination            0
country                0
latitude               0
longitude              0
temperature            0
feels_like             0
humidity               0
pressure               0
wind_speed             0
cloudiness             0
weather_condition      0
weather_description    0
visibility             2
rain_1h                0
timestamp              0
dtype: int64


In [67]:
# Fill the small number of missing visibility values.
# We use the median because visibility can vary considerably
# and the median is less affected by unusually high values.

visibility_median = weather_complete["visibility"].median()

weather_complete["visibility"] = weather_complete["visibility"].fillna(
    visibility_median
)

print("Visibility median used:", visibility_median)
print("Missing visibility:", weather_complete["visibility"].isna().sum())

Visibility median used: 10000.0
Missing visibility: 0


In [68]:
# Final validation of the complete weather dataset.
# We confirm that all 50 destinations are present and
# there are no missing values remaining.

print("Rows:", len(weather_complete))
print("Unique destinations:", weather_complete["destination"].nunique())
print("Duplicate destinations:",
      weather_complete["destination"].duplicated().sum())

print("\nMissing values:")
print(weather_complete.isnull().sum())

Rows: 50
Unique destinations: 50
Duplicate destinations: 0

Missing values:
destination            0
country                0
latitude               0
longitude              0
temperature            0
feels_like             0
humidity               0
pressure               0
wind_speed             0
cloudiness             0
weather_condition      0
weather_description    0
visibility             0
rain_1h                0
timestamp              0
dtype: int64


In [69]:
# Save the validated 50-destination weather dataset.
# This prevents us from needing to call the Weather API again.

weather_complete.to_csv(
    "../data/cleaned/weather_complete.csv",
    index=False
)

print("Complete weather dataset saved successfully.")

Complete weather dataset saved successfully.


In [83]:
# Reload the saved weather dataset to confirm
# that the file was written correctly.

weather_check = pd.read_csv(
    "../data/cleaned/weather_complete.csv"
)

print("Rows:", len(weather_check))
print("Unique destinations:", weather_check["destination"].nunique())
print("Missing values:", weather_check.isnull().sum().sum())

Rows: 9
Unique destinations: 9
Missing values: 1


In [71]:
# Merge the destination master data with the complete weather dataset.
# We use a LEFT JOIN so that every destination from the master dataset
# must remain in the final dataset.

destination_weather = destination_master.merge(
    weather_complete,
    on="destination",
    how="left",
    validate="one_to_one"
)

print("Rows:", len(destination_weather))
print("Unique destinations:", destination_weather["destination"].nunique())

Rows: 50
Unique destinations: 50


In [72]:
# Check whether any destination failed to receive weather information.
# A valid merge should have zero missing weather temperatures.

print("Missing temperature:",
      destination_weather["temperature"].isna().sum())

print("Missing weather condition:",
      destination_weather["weather_condition"].isna().sum())

Missing temperature: 4
Missing weather condition: 4


In [73]:
# Show the destinations that still have missing weather data.
# This helps us identify exactly which destinations failed during the merge.

missing_weather = destination_weather[
    destination_weather["temperature"].isna()
]

print("Destinations with missing weather:")
print(missing_weather["destination"].tolist())

print("\nNumber of missing destinations:",
      len(missing_weather))

Destinations with missing weather:
['Kodaikanal', 'Thiruvananthapuram', 'Pondicherry', 'Pahalgam']

Number of missing destinations: 4


In [74]:
# Get coordinates for the 4 destinations whose weather data is missing.
# We will use these coordinates to request weather directly instead of
# relying on the destination name being recognized by OpenWeather.

missing_destinations = [
    "Kodaikanal",
    "Thiruvananthapuram",
    "Pondicherry",
    "Pahalgam"
]

missing_locations = destination_locations[
    destination_locations["destination"].isin(missing_destinations)
][["destination", "latitude", "longitude"]]

print(missing_locations)

           destination   latitude  longitude
16          Kodaikanal  10.233712  77.491972
21  Thiruvananthapuram   8.488227  76.947551
23         Pondicherry  11.934057  79.830645
49            Pahalgam  34.032205  75.322648


In [77]:
# Collect weather for the 4 destinations using their coordinates.
# Passing the destination name is required by our existing function.

missing_weather_records = []

for _, row in missing_locations.iterrows():

    destination = row["destination"]
    latitude = row["latitude"]
    longitude = row["longitude"]

    print(f"Collecting weather by coordinates: {destination}")

    try:
        # Pass all three required arguments:
        # destination, latitude, and longitude
        weather = get_weather_by_coordinates(
            destination,
            latitude,
            longitude
        )

        if weather is not None:
            missing_weather_records.append(weather)

    except Exception as e:
        print(f"API request failed for {destination}: {e}")

# Convert collected records into a DataFrame
missing_weather_df = pd.DataFrame(missing_weather_records)

print("\nNew weather records collected:", len(missing_weather_df))

missing_weather_df


New weather records collected: 4


,destination,country,latitude,longitude,temperature,feels_like,humidity,pressure,wind_speed,cloudiness,weather_condition,weather_description,visibility,rain_1h,timestamp
0,Kodaikanal,IN,10.2337,77.4920,14.75,14.47,84,1015,1.78,100,Clouds,overcast clouds,10000,0.00,2026-08-25T15:59:33.022947+00:00
1,Thiruvananthapuram,IN,8.4882,76.9476,26.94,30.46,89,1014,3.09,100,Clouds,overcast clouds,10000,0.00,2026-08-25T15:59:33.772839+00:00
2,Pondicherry,IN,11.9341,79.8306,28.19,31.04,70,1009,6.17,100,Rain,light rain,10000,0.25,2026-08-25T15:59:34.892180+00:00
3,Pahalgam,IN,34.0322,75.3226,16.53,16.09,71,1015,2.25,0,Clear,clear sky,10000,0.00,2026-08-25T15:59:38.153311+00:00


In [78]:
# Combine the existing weather data with the 4 newly collected records.
# This gives us the complete weather dataset for all 50 destinations.

weather_complete = pd.concat(
    [weather_data, missing_weather_df],
    ignore_index=True
)

# Remove any accidental duplicate destinations.
weather_complete = weather_complete.drop_duplicates(
    subset="destination",
    keep="last"
)

print("Rows:", len(weather_complete))
print("Unique destinations:", weather_complete["destination"].nunique())

Rows: 9
Unique destinations: 9


In [79]:
# Check whether any weather values are still missing
# after adding the 4 coordinate-based records.

print("Missing values:")
print(weather_complete.isna().sum())

Missing values:
destination            0
country                0
latitude               0
longitude              0
temperature            0
feels_like             0
humidity               0
pressure               0
wind_speed             0
cloudiness             0
weather_condition      0
weather_description    0
visibility             1
rain_1h                0
timestamp              0
dtype: int64


In [80]:
# Save the complete weather dataset so we don't need to call
# the weather API again when reopening VS Code.

weather_complete.to_csv(
    "../data/cleaned/weather_complete.csv",
    index=False
)

print("Weather dataset saved successfully.")
print("Rows:", len(weather_complete))
print("Unique destinations:", weather_complete["destination"].nunique())

Weather dataset saved successfully.
Rows: 9
Unique destinations: 9


In [81]:
# Load the already collected weather data.
# This avoids unnecessary API calls when reopening the notebook.

weather_complete = pd.read_csv(
    "../data/cleaned/weather_complete.csv"
)

print("Rows:", len(weather_complete))
print("Unique destinations:", weather_complete["destination"].nunique())

Rows: 9
Unique destinations: 9


In [82]:
# Load the saved weather file into a fresh variable.
# Using a new variable prevents accidental overwriting.

weather_saved = pd.read_csv(
    "../data/cleaned/weather_complete.csv"
)

print("Rows:", len(weather_saved))
print("Unique destinations:", weather_saved["destination"].nunique())

print("\nDestinations:")
print(weather_saved["destination"].tolist())

Rows: 9
Unique destinations: 9

Destinations:
['Hyderabad', 'Goa', 'Munnar', 'Manali', 'Jaipur', 'Kodaikanal', 'Thiruvananthapuram', 'Pondicherry', 'Pahalgam']


In [84]:
# Check the weather-related DataFrame variables currently available.
# We are looking for the DataFrame containing all 50 destinations.

print("weather_data:")
print("Rows:", len(weather_data))
print("Unique destinations:", weather_data["destination"].nunique())

print("\nmissing_weather_df:")
print("Rows:", len(missing_weather_df))
print("Unique destinations:", missing_weather_df["destination"].nunique())

weather_data:
Rows: 5
Unique destinations: 5

missing_weather_df:
Rows: 4
Unique destinations: 4


In [86]:
# Take a snapshot of the current notebook variables first.
# This prevents "dictionary changed size during iteration".

all_variables = list(globals().items())

for name, value in all_variables:
    if isinstance(value, pd.DataFrame):
        if "destination" in value.columns:
            if len(value) in [38, 43, 46, 50]:
                print(f"{name}: {value.shape}")

destination_master: (50, 3)
location_df: (50, 15)
_27: (50, 10)
places_feature_df: (50, 12)
places_model_df: (50, 27)
capped_flags: (50, 5)
places_model_features: (50, 16)
places_check: (50, 16)
new_weather_data: (38, 15)
destination_locations: (50, 15)
destination_weather: (50, 17)


In [87]:
# Check the existing 50-destination weather dataset.
# This confirms that all destinations are present before we overwrite
# the incorrectly saved weather_complete.csv file.

print("Rows:", len(destination_weather))
print("Unique destinations:", destination_weather["destination"].nunique())

print("\nMissing values:")
print(destination_weather.isna().sum())

Rows: 50
Unique destinations: 50

Missing values:
destination_id         0
destination            0
search_query           0
country                4
latitude               4
longitude              4
temperature            4
feels_like             4
humidity               4
pressure               4
wind_speed             4
cloudiness             4
weather_condition      4
weather_description    4
visibility             4
rain_1h                4
timestamp              4
dtype: int64


In [88]:
# Start with the existing 50-destination master + weather dataset.
# The four missing destinations currently have NaN weather values.

# Remove the incomplete weather columns from the current merged dataset.
# We will replace them with the correct values from missing_weather_df.

weather_columns = [
    "country",
    "latitude",
    "longitude",
    "temperature",
    "feels_like",
    "humidity",
    "pressure",
    "wind_speed",
    "cloudiness",
    "weather_condition",
    "weather_description",
    "visibility",
    "rain_1h",
    "timestamp"
]

# Keep only the destination information from the current 50-row dataset.
destination_base = destination_weather[
    ["destination_id", "destination", "search_query"]
].copy()

# Add the complete weather records we have collected.
destination_weather_fixed = destination_base.merge(
    pd.concat(
        [
            new_weather_data,
            missing_weather_df
        ],
        ignore_index=True
    ),
    on="destination",
    how="left"
)

print("Rows:", len(destination_weather_fixed))
print("Unique destinations:",
      destination_weather_fixed["destination"].nunique())

print("\nMissing values:")
print(destination_weather_fixed[weather_columns].isna().sum())

Rows: 50
Unique destinations: 50

Missing values:
country                12
latitude               12
longitude              12
temperature            12
feels_like             12
humidity               12
pressure               12
wind_speed             12
cloudiness             12
weather_condition      12
weather_description    12
visibility             12
rain_1h                12
timestamp              12
dtype: int64


In [89]:
# Compare the destination names available in each weather DataFrame.
# This lets us identify exactly which destinations are missing or mismatched
# before performing the final merge.

master_destinations = set(destination_master["destination"])

weather_5 = set(weather_data["destination"])
weather_38 = set(new_weather_data["destination"])
weather_4 = set(missing_weather_df["destination"])

all_collected = weather_5 | weather_38 | weather_4

print("Destination master:", len(master_destinations))
print("Original weather:", len(weather_5))
print("New weather:", len(weather_38))
print("Coordinate weather:", len(weather_4))
print("Combined unique weather destinations:", len(all_collected))

print("\nMissing from collected weather:")
print(sorted(master_destinations - all_collected))

print("\nExtra weather destinations not in master:")
print(sorted(all_collected - master_destinations))

Destination master: 50
Original weather: 5
New weather: 38
Coordinate weather: 4
Combined unique weather destinations: 47

Missing from collected weather:
['Coorg', 'Dharamshala', 'Jim Corbett', 'Kaziranga', 'Ladakh', 'Ranthambore', 'Wayanad']

Extra weather destinations not in master:
['Kodaikānāl', 'Pahlgām', 'Puducherry', 'Trivandrum']


In [90]:
# Search all current DataFrames for the 7 destinations that are missing.
# We are looking for the coordinate-based weather records collected earlier.

target_destinations = {
    "Coorg",
    "Dharamshala",
    "Jim Corbett",
    "Kaziranga",
    "Ladakh",
    "Ranthambore",
    "Wayanad"
}

for name, value in list(globals().items()):

    if isinstance(value, pd.DataFrame) and "destination" in value.columns:

        matches = value[
            value["destination"].astype(str).isin(target_destinations)
        ]

        if len(matches) > 0:
            print(f"\n{name}: {len(matches)} matching records")
            print(matches["destination"].tolist())


__: 1 matching records
['Ranthambore']

destination_master: 7 matching records
['Coorg', 'Wayanad', 'Ladakh', 'Dharamshala', 'Kaziranga', 'Jim Corbett', 'Ranthambore']

_23: 1 matching records
['Ranthambore']

location_df: 7 matching records
['Coorg', 'Wayanad', 'Ladakh', 'Dharamshala', 'Kaziranga', 'Jim Corbett', 'Ranthambore']

_27: 7 matching records
['Coorg', 'Wayanad', 'Ladakh', 'Dharamshala', 'Kaziranga', 'Jim Corbett', 'Ranthambore']

places_feature_df: 7 matching records
['Coorg', 'Dharamshala', 'Jim Corbett', 'Kaziranga', 'Ladakh', 'Ranthambore', 'Wayanad']

places_raw_df: 475 matching records
['Coorg', 'Coorg', 'Coorg', 'Coorg', 'Coorg', 'Coorg', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Wayanad', 'Way

In [91]:
# Normalize destination names so they exactly match destination_master.
# This prevents the same destination from appearing under different names.

name_mapping = {
    "Kodaikānāl": "Kodaikanal",
    "Pahlgām": "Pahalgam",
    "Puducherry": "Pondicherry",
    "Trivandrum": "Thiruvananthapuram"
}

weather_data = weather_data.copy()
new_weather_data = new_weather_data.copy()
coordinate_weather_data = coordinate_weather_data.copy()

for df in [weather_data, new_weather_data, coordinate_weather_data]:
    df["destination"] = df["destination"].replace(name_mapping)

In [92]:
# Combine all weather records collected from the three sources.
# ignore_index=True creates a clean continuous index.

all_weather = pd.concat(
    [
        weather_data,
        new_weather_data,
        coordinate_weather_data
    ],
    ignore_index=True
)

print("Total weather records:", len(all_weather))
print("Unique destinations:", all_weather["destination"].nunique())

Total weather records: 50
Unique destinations: 50


In [93]:
# Verify that every destination in the master dataset
# has exactly one corresponding weather record.

master_destinations = set(destination_master["destination"])
weather_destinations = set(all_weather["destination"])

print("Missing from weather:")
print(sorted(master_destinations - weather_destinations))

print("\nExtra destinations in weather:")
print(sorted(weather_destinations - master_destinations))

print("\nDuplicate weather destinations:")
print(
    all_weather[
        all_weather["destination"].duplicated(keep=False)
    ]["destination"].unique()
)

Missing from weather:
[]

Extra destinations in weather:
[]

Duplicate weather destinations:
<StringArray>
[]
Length: 0, dtype: str


In [94]:
# Merge the validated weather data with the places-based model features.
# A left join keeps all 50 destinations from the places dataset.

travel_features = places_model_features.merge(
    all_weather,
    on="destination",
    how="left",
    validate="one_to_one"
)

print("Rows:", len(travel_features))
print("Unique destinations:", travel_features["destination"].nunique())
print("Columns:", len(travel_features.columns))

Rows: 50
Unique destinations: 50
Columns: 30


In [95]:
# Verify that the merge did not create missing weather values.

weather_columns = [
    "country",
    "latitude",
    "longitude",
    "temperature",
    "feels_like",
    "humidity",
    "pressure",
    "wind_speed",
    "cloudiness",
    "weather_condition",
    "weather_description",
    "visibility",
    "rain_1h",
    "timestamp"
]

print("Missing weather values:")
print(travel_features[weather_columns].isna().sum())

Missing weather values:
country                0
latitude               0
longitude              0
temperature            0
feels_like             0
humidity               0
pressure               0
wind_speed             0
cloudiness             0
weather_condition      0
weather_description    0
visibility             2
rain_1h                0
timestamp              0
dtype: int64


In [96]:
# Identify the destinations where visibility is missing.
# We will inspect them before deciding how to handle the missing values.

travel_features[
    travel_features["visibility"].isna()
][["destination", "visibility"]]

,destination,visibility
9,Coorg,NaN
31,Munnar,NaN


In [97]:
# Fill the two missing visibility values using the median
# visibility across all destinations with available data.
# This keeps all 50 destinations in the final dataset.

visibility_median = travel_features["visibility"].median()

travel_features["visibility"] = travel_features["visibility"].fillna(
    visibility_median
)

print("Visibility median used:", visibility_median)

print("\nMissing visibility values:")
print(travel_features["visibility"].isna().sum())

Visibility median used: 10000.0

Missing visibility values:
0


In [98]:
# Final validation of the merged travel feature dataset.

print("Rows:", len(travel_features))
print("Unique destinations:", travel_features["destination"].nunique())

print("\nTotal missing values:")
print(travel_features.isna().sum())

print("\nDuplicate destinations:")
print(
    travel_features[
        travel_features["destination"].duplicated(keep=False)
    ]["destination"].unique()
)

Rows: 50
Unique destinations: 50

Total missing values:
destination             0
sight_score             0
park_score              0
restaurant_score        0
water_score             0
forest_score            0
wetland_score           0
river_score             0
mountain_score          0
coastal_score           0
sand_score              0
protected_area_score    0
sights_capped           0
park_capped             0
restaurant_capped       0
natural_capped          0
country                 0
latitude                0
longitude               0
temperature             0
feels_like              0
humidity                0
pressure                0
wind_speed              0
cloudiness              0
weather_condition       0
weather_description     0
visibility              0
rain_1h                 0
timestamp               0
dtype: int64

Duplicate destinations:
<StringArray>
[]
Length: 0, dtype: str


In [99]:
# Final validation of the complete travel feature dataset.
# This confirms that we have all 50 destinations and no missing values.

print("Rows:", len(travel_features))
print("Unique destinations:", travel_features["destination"].nunique())

print("\nTotal missing values:")
print(travel_features.isna().sum())

print("\nDuplicate destinations:")
print(
    travel_features[
        travel_features["destination"].duplicated(keep=False)
    ]["destination"].unique()
)

Rows: 50
Unique destinations: 50

Total missing values:
destination             0
sight_score             0
park_score              0
restaurant_score        0
water_score             0
forest_score            0
wetland_score           0
river_score             0
mountain_score          0
coastal_score           0
sand_score              0
protected_area_score    0
sights_capped           0
park_capped             0
restaurant_capped       0
natural_capped          0
country                 0
latitude                0
longitude               0
temperature             0
feels_like              0
humidity                0
pressure                0
wind_speed              0
cloudiness              0
weather_condition       0
weather_description     0
visibility              0
rain_1h                 0
timestamp               0
dtype: int64

Duplicate destinations:
<StringArray>
[]
Length: 0, dtype: str


In [100]:
# Save the complete travel feature dataset.
# This file combines places-based features with weather information
# for all 50 destinations.

output_path = "../data/cleaned/travel_features.csv"

travel_features.to_csv(output_path, index=False)

print("Travel features dataset saved to:")
print(output_path)

Travel features dataset saved to:
../data/cleaned/travel_features.csv


# Accommodation Data Collection

## Objective

Collect accommodation information for all 50 initial destinations.

The accommodation data will later be used to:
- show available accommodation options
- estimate accommodation cost
- compare budget, mid-range, and higher-priced stays
- calculate the total trip budget
- support the final travel plan

## Input

We will use the existing destination master and location data:

- `../data/raw/destination_master.csv`
- `../data/raw/places/destination_locations.csv`

These already contain the 50 destinations and their geographic information.

## Collection Flow

Destination Master
        ↓
Destination Coordinates
        ↓
Accommodation API
        ↓
Raw Accommodation Records
        ↓
Data Validation
        ↓
Data Cleaning
        ↓
Destination-level Accommodation Features
        ↓
Accommodation Cost Features

## Important

We are keeping the initial scope at 50 destinations.

The collection code should be written so that later we can expand
the same approach to broader coverage across India without changing
the overall pipeline.

## Expected Raw Output

`../data/raw/accommodation/accommodation_raw.csv`

## Expected Cleaned Output

`../data/cleaned/accommodation_cleaned.csv`

## Expected Feature Output

`../data/cleaned/accommodation_features.csv`

## After this approach is completely validated

We will move to the next approach:

Accommodation → Flights